# Leader → Follower 명령 생성기: C++ 입문자를 위한 코드 독해 노트

**학습 대상:** `joint_trajectory_command_broadcaster.cpp` — 사용자 첨부 원문 1,432줄.

이 파일은 사람이 움직이는 Leader의 관절 위치를 읽고, Follower에 전달할 `JointTrajectory` 목표를 만드는 **ros2_control 플러그인**입니다. 독립 실행 프로그램의 `main()` 대신 controller manager가 생명주기 함수와 `update()`를 호출합니다.

이 노트는 다음을 한 파일 안에 담았습니다.

1. C++ 문법 사전과 전체 데이터 흐름
2. 원문 전체에 한국어 학습 주석을 추가한 코드
3. 각 구간의 입력·출력·호출 관계와 읽기 포인트
4. 보간·상대 이동·그리퍼 동작을 확인하는 작은 계산 예제
5. 복습 문제, 정답, 구현을 읽을 때 구분해야 할 사항

**사용 방법:** VS Code 또는 Jupyter에서 이 `.ipynb`를 엽니다. C++ 원문은 Markdown 코드 블록으로 표시하므로 C++ 커널 없이 읽을 수 있습니다. Python 코드 셀 네 개는 숫자 계산만 수행하며 ROS나 로봇에 연결하지 않습니다. 계산 결과도 저장되어 있어 실행 없이 볼 수 있습니다.

**표기:** `// [학습]`은 이 노트를 위해 추가한 주석입니다. 기존 영어 주석은 원문 그대로 남겼습니다. 각 구간에 붙은 원문 줄 번호는 추가 주석을 제외한 첨부 파일 기준입니다. 원본 `.cpp`는 수정하지 않았습니다.

**읽는 순서:** 처음에는 문법 사전 → 전체 흐름 → §13~§20의 `update()`와 모드 전환 → §7~§8의 콜백 순서로 읽으세요. §21~§23의 기구학은 두 번째 읽기에서 보아도 됩니다.

## 0. 이 코드의 위치와 주변 구성

```text
Leader 하드웨어의 position 상태 인터페이스
        ↓
controller manager → update(time, period)
        ↓
JointTrajectoryCommandBroadcaster
        ├─ IDLE: follower 측정값에 내부 목표 맞추기
        ├─ TELEOP: leader 움직임으로 목표 계산
        └─ SAVE_POSE: 저장 자세까지 보간
        ↓
왼팔 / 오른팔 JointTrajectory 발행
        ↓
launch의 namespace·remap에 따라 명령 경로 결정
        ↓
현재 Mini Leader AI 기본 구성: intervention teleop 입력 → arbiter → follower 제어기
```

이 `.cpp` 자체에는 AI 추론이나 arbiter의 소스 선택 로직이 없습니다. 메시지를 발행했다는 사실과 그 메시지가 최종 로봇 명령으로 선택되었다는 사실도 구분해야 합니다.

| 주체 | 담당 |
|---|---|
| `controller_manager` | 플러그인의 생명주기 관리, 주기적 `update()` 호출 |
| 이 broadcaster | Leader 입력을 Follower 목표 메시지로 변환 |
| `command_source_arbiter` | 별도 구성에서 policy/teleop 등의 최종 명령 소스 선택 |
| follower의 `JointTrajectoryController` | 받은 목표를 하드웨어 제어 명령으로 처리 |
| `joint_trajectory_executor` | 초기 자세 등 정해진 궤적을 액션으로 요청하는 별도 실행기 |

`gripper_trigger.cpp`는 같은 패키지에 있지만 첨부한 파일과는 별도의 실행 프로그램입니다.

**전체 생명주기**

```text
on_init() → on_configure() → on_activate()
                                  ↓
                          update() 반복 호출
                                  ↓
                          on_deactivate()
```

콜백은 메시지 수신 시 실행됩니다. 이 코드에서는 콜백이 버퍼에 데이터를 넣고 `update()`가 그 버퍼를 읽는 연결이 중요합니다. C++ 소스의 위에서 아래로 함수들이 자동 실행되는 구조가 아닙니다.

## C++ 문법 사전 — 이 코드에서 자주 만나는 표기

| 표기 | 입문자 관점의 뜻 | 이 코드의 예 |
|---|---|---|
| `#include` | 다른 파일에 선언된 타입·함수를 사용하기 위한 헤더 포함 | `#include <vector>` |
| `namespace` / `::` | 이름의 소속 범위 / 그 범위 안의 이름 지정 | `std::vector`, `ClassName::update` |
| `class`, `struct` | 데이터와 함수를 묶는 사용자 정의 타입 | `GroupPoseKinematics` |
| `auto` | 초기값으로 변수 타입을 컴파일러가 추론 | `auto it = map.find(key);` |
| `const T & x` | 복사 없이 원본을 읽는 참조 | `const std::string & group_name` |
| `auto & x` | 원본을 수정할 수 있는 참조 | `auto & rt = group_runtime_[group_name]` |
| `T * p` / `nullptr` | 객체를 가리키는 포인터 / 가리킬 대상 없음 | `KDL::JntArray * joint_array_out` |
| `obj.member` / `ptr->member` | 객체의 멤버 접근 / 포인터가 가리키는 객체의 멤버 접근 | `msg.position`, `msg->position` |
| `*p` / `&x` | 포인터가 가리키는 객체 / 객체의 주소 | `*follower_positions`, `&current_joints` |
| `std::vector<T>` | 크기를 바꿀 수 있는 배열 | 관절 위치 목록 |
| `std::unordered_map<K,V>` | 키로 값을 찾는 해시 테이블 | 그룹 이름 → 목표 배열 |
| `shared_ptr` | 여러 소유자가 수명을 공유하는 스마트 포인터 | publisher, buffer |
| `unique_ptr` | 하나의 소유자가 관리하는 스마트 포인터 | FK solver |
| `make_shared`, `make_unique` | 객체 생성과 스마트 포인터 구성을 함께 수행 | `make_shared<ParamListener>(...)` |
| `template<typename T>` | 여러 타입에 재사용 가능한 함수·클래스 | `has_any_key()` |
| `[this, group_name](...) { ... }` | 이름 없는 함수인 람다; 대괄호는 외부 값 캡처 | 구독 콜백 |
| `?:` | 조건에 따라 둘 중 하나 선택 | `enabled ? TELEOP : IDLE` |
| `&&`, `||`, `!` | 그리고, 또는, 부정. `&&`·`||`는 단축 평가 | 포인터가 있는지 먼저 검사 |
| `static_cast<T>(x)` | 의도한 타입 변환을 명시 | 관절 인덱스의 정수 타입 변환 |
| `std::move(x)` | 이동 가능한 값으로 취급하도록 표시 | 컨테이너 소유 데이터 전달 |
| `return`, `break`, `continue` | 함수 종료 / 가까운 반복문·switch 종료 / 다음 반복으로 |
| `try` / `catch` | 예외가 발생한 경우 처리 | 파라미터 초기화 실패 |
| `enum class` | 범위가 구분되는 열거형 | `Mode::TELEOP` |
| 이름 끝의 `_` | 멤버 변수임을 표시하는 코딩 관례 | `params_`, `group_runtime_` |

### 참조를 복사와 구별하기

```cpp
auto copy = group_last_target_["left"];     // 별도 vector 복사
auto & ref = group_last_target_["left"];    // 기존 vector의 다른 이름
ref[0] = 0.5;                              // map 안의 값도 바뀜
const auto & view = group_last_target_["left"]; // 읽기 전용 참조
```

### map을 읽는 세 가지 방식

- `map[key]`: 키가 없으면 기본값을 **삽입**합니다. 읽는 것처럼 보여도 내용이 변할 수 있습니다.
- `map.find(key)`: 못 찾으면 `map.end()`를 반환합니다. 삽입하지 않습니다.
- `map.at(key)`: 키가 없으면 예외가 발생합니다.

### 람다를 읽는 순서

```cpp
[this, group_name](sensor_msgs::msg::JointState::SharedPtr msg) {
  // this: 현재 객체 접근, group_name: 그 시점 문자열의 복사본을 캡처
  // msg: 콜백을 호출할 때 전달되는 메시지 스마트 포인터
};
```

`[&x]`는 참조 캡처입니다. 콜백이 실행될 때까지 `x`가 살아 있어야 합니다. `this` 캡처도 객체 전체를 복사하는 것이 아니라 현재 객체의 포인터를 사용합니다.

## 핵심 데이터 사전

| 변수 | 저장 내용 | 읽을 때 기억할 점 |
|---|---|---|
| `params_` | 설정 파라미터 묶음 | YAML 스키마로 생성된 타입 사용 |
| `state_interfaces_` | ros2_control이 제공한 Leader 하드웨어 상태 핸들 | Leader 위치의 주요 입력 |
| `name_if_value_mapping_` | 관절 이름 → 인터페이스 이름 → 값 | 예: `["arm_l_joint1"]["position"]` |
| `group_joint_names_` | 그룹별 관절 순서 | 메시지 배열 순서의 기준 |
| `group_last_target_` | 그룹별 마지막 유효 목표 | **실측값과 동일하다고 가정하지 않기** |
| `group_runtime_` | 그룹별 모드·보간·그리퍼 기준 상태 | 왼팔과 오른팔이 각각 보유 |
| `group_follower_state_buffers_` | follower 실측 관절 상태 | 콜백 → update 전달 |
| `group_executed_action_buffers_` | 최종 전달된 명령의 복사본 | 그리퍼 목표 보존에 사용 |
| `group_enable_command_buffers_` | 최근 모드 변경 명령과 일련번호 | 모든 명령을 보존하는 FIFO 큐가 아님 |
| `group_pose_kinematics_` | FK·Jacobian·상대 기준 자세 | 상대 모드에서 사용 |

`JointState`는 `name`과 `position` 등 배열로 관절 상태를 표현합니다. `JointTrajectory`는 `joint_names`와 하나 이상의 `points`로 목표를 표현합니다. 이 파일의 궤적 출력은 **점 하나를 반복 갱신하는 형태**입니다.

원문 외에 확인한 보조 파일: `joint_trajectory_command_broadcaster.hpp`, `relative_pose_mapper.hpp`, `leader_action_latch.hpp`, `toggle_command_gate.hpp`, 파라미터 YAML. 뒤의 수학·그리퍼 설명은 이 보조 구현을 함께 확인한 내용입니다.

## §1. 헤더·namespace·초기값

**원문 1~58행**

**목표:** C++ 파일의 준비 부분을 읽습니다. `#include`는 선언을 가져오고, `namespace`는 이름을 묶습니다. `quiet_NaN()`은 아직 값이 없다는 표시이며 숫자 0과 다릅니다. `std::isfinite()`로 정상 수치인지 확인하는 흐름이 나중에 반복됩니다.

`rclcpp_lifecycle::State`는 클래스 전방 선언입니다. 객체를 생성하거나 함수를 호출하는 문장이 아닙니다.

```cpp
// Copyright 2021 ros2_control development team
//
// Licensed under the Apache License, Version 2.0 (the "License");
// you may not use this file except in compliance with the License.
// You may obtain a copy of the License at
//
//     http://www.apache.org/licenses/LICENSE-2.0
//
// Unless required by applicable law or agreed to in writing, software
// distributed under the License is distributed on an "AS IS" BASIS,
// WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
// See the License for the specific language governing permissions and
// limitations under the License.

// [학습] 프로젝트 헤더: 클래스 선언과 멤버 변수는 .hpp에, 함수 본문은 이 .cpp에 있습니다.
#include "joint_trajectory_command_broadcaster/joint_trajectory_command_broadcaster.hpp"
// [학습] 그리퍼 판별·유효 목표 저장·그리퍼 유지 관련 보조 함수가 선언된 헤더입니다.
#include "joint_trajectory_command_broadcaster/leader_action_latch.hpp"

// [학습] 표준 라이브러리 헤더 묶음입니다. <...> 안은 헤더 이름이며 함수 호출이 아닙니다.
#include <chrono>
#include <cstddef>
#include <limits>
#include <memory>
#include <string>
#include <thread>
#include <unordered_map>
#include <vector>
#include <functional>
#include <cmath>
#include <algorithm>
#include <iterator>

// [학습] Eigen은 행렬 계산, KDL은 로봇 체인의 순기구학과 Jacobian 계산에 사용합니다.
#include <Eigen/Core>
#include <kdl/chainfksolverpos_recursive.hpp>
#include <kdl/chainjnttojacsolver.hpp>
#include <kdl/jacobian.hpp>
#include <kdl/jntarray.hpp>
#include <kdl_parser/kdl_parser.hpp>

#include "hardware_interface/types/hardware_interface_type_values.hpp"
#include "rclcpp/qos.hpp"
#include "rclcpp/time.hpp"
#include "std_msgs/msg/header.hpp"
#include "std_msgs/msg/string.hpp"
#include "trajectory_msgs/msg/joint_trajectory.hpp"
#include "sensor_msgs/msg/joint_state.hpp"
#include "urdf/model.h"

// [학습] 전방 선언: State라는 클래스가 있다는 사실만 알립니다. 본문 정의는 다른 헤더가 제공합니다.
namespace rclcpp_lifecycle
{
class State;
}  // namespace rclcpp_lifecycle

// [학습] 이 범위 안의 클래스·함수 이름을 같은 namespace로 묶어 이름 충돌을 줄입니다.
namespace joint_trajectory_command_broadcaster
{
// [학습] NaN은 아직 정상 수치가 없는 상태를 표시합니다. 0은 정상 관절 위치일 수 있어 구별해야 합니다.
const auto kUninitializedValue = std::numeric_limits<double>::quiet_NaN();
// [학습] using 선언으로 긴 이름 대신 HW_IF_POSITION처럼 짧은 이름을 사용할 수 있습니다.
using hardware_interface::HW_IF_EFFORT;
using hardware_interface::HW_IF_POSITION;
using hardware_interface::HW_IF_VELOCITY;
}

```

## §2. 운동학 데이터 구조와 생성자

**원문 59~85행**

**입력:** Leader와 Follower의 KDL 관절 체인. **저장:** 체인별 FK 계산기, Jacobian 계산기, 관절 인덱스 대응표.

생성자의 `: leader(leader_chain), follower(follower_chain)`은 멤버 초기화 목록입니다. `unique_ptr`는 계산기 수명을 자동 관리합니다. 그룹 배열에 8개 관절이 있고 손끝 체인에는 팔 관절 7개만 있을 수 있어 별도 인덱스 표가 필요합니다.

```cpp
// [학습] 클래스 내부에 선언된 중첩 struct의 실제 정의입니다. struct의 기본 접근 권한은 public입니다.
struct JointTrajectoryCommandBroadcaster::GroupPoseKinematics
{
  struct ChainKinematics
  {
    // [학습] explicit은 의도하지 않은 암시적 생성자 변환을 막습니다. const &는 체인을 복사 없이 인자로 받습니다.
    explicit ChainKinematics(const KDL::Chain & configured_chain)
    // [학습] 콜론 뒤는 멤버 초기화 목록입니다. 본문 {} 실행 전에 멤버를 생성·초기화합니다.
    : chain(configured_chain),
      // [학습] unique_ptr로 계산기를 소유합니다. 소유 객체가 파괴될 때 계산기도 자동 정리됩니다.
      fk_solver(std::make_unique<KDL::ChainFkSolverPos_recursive>(chain)),
      jacobian_solver(std::make_unique<KDL::ChainJntToJacSolver>(chain))
    {
    }

    KDL::Chain chain;
    std::unique_ptr<KDL::ChainFkSolverPos_recursive> fk_solver;
    std::unique_ptr<KDL::ChainJntToJacSolver> jacobian_solver;
    // [학습] KDL 관절 순서와 그룹 배열 순서가 다를 수 있어 대응 인덱스를 저장합니다.
    std::vector<size_t> group_joint_indices;
  };

  GroupPoseKinematics(
    const KDL::Chain & leader_chain, const KDL::Chain & follower_chain)
  : leader(leader_chain), follower(follower_chain)
  {
  }

  ChainKinematics leader;
  ChainKinematics follower;
  RelativePoseMapper relative_mapper;
};
```

## §3. compute_pose(): 관절 위치를 손끝 자세로 바꾸기

**원문 86~125행**

**입력:** 관절 위치 배열. **출력:** `pose` 참조와 선택적 `joint_array_out` 포인터. **반환값:** 성공/실패 `bool`.

처리 순서는 크기 검사 → KDL 순서로 배열 재배열 → FK → KDL 결과를 Eigen의 회전·이동으로 복사입니다. `return false`는 오류를 호출자에게 알리며, 호출자는 마지막 유효 목표를 유지할 수 있습니다. `pose`는 목표 관절값이 아니라 손끝의 공간상 위치·방향입니다.

```cpp

// [학습] 이름 없는 namespace: 이 번역 단위(.cpp) 내부에서 쓰는 보조 함수를 둡니다.
namespace
{
// [학습] 함수 템플릿: 필요한 멤버를 가진 서로 다른 KinematicsT 타입에 같은 로직을 적용합니다.
template<typename KinematicsT>
bool compute_pose(
  KinematicsT & kinematics,
  const std::vector<double> & group_position,
  // [학습] pose는 출력용 참조입니다. 함수 안에서 수정하면 호출자의 pose도 바뀝니다.
  Eigen::Isometry3d & pose,
  // [학습] 선택적 출력 포인터입니다. nullptr이면 관절 배열 결과를 별도로 돌려주지 않습니다.
  KDL::JntArray * joint_array_out = nullptr)
{
  // [학습] 계산 전에 관절 개수가 맞는지 검사하는 조기 반환 패턴입니다.
  if (kinematics.group_joint_indices.size() != kinematics.chain.getNrOfJoints()) {
    return false;
  }
  KDL::JntArray joints(kinematics.chain.getNrOfJoints());
  for (size_t i = 0; i < kinematics.group_joint_indices.size(); ++i) {
    const size_t group_index = kinematics.group_joint_indices[i];
    // [학습] 배열 범위를 먼저 확인하고, 값이 NaN/무한대인지 검사합니다. ||는 앞이 참이면 뒤를 생략합니다.
    if (group_index >= group_position.size() || !std::isfinite(group_position[group_index])) {
      return false;
    }
    joints(static_cast<unsigned int>(i)) = group_position[group_index];
  }

  // [학습] 관절 위치로 손끝 자세를 구하는 FK입니다. KDL Frame은 회전과 이동을 함께 표현합니다.
  KDL::Frame frame;
  if (kinematics.fk_solver->JntToCart(joints, frame) < 0) {
    return false;
  }
  // [학습] 4x4 강체변환을 단위행렬로 초기화하고 이동·회전을 채웁니다.
  pose.setIdentity();
  for (size_t row = 0; row < 3; ++row) {
    pose.translation()(static_cast<Eigen::Index>(row)) = frame.p[static_cast<unsigned int>(row)];
    for (size_t col = 0; col < 3; ++col) {
      pose.linear()(static_cast<Eigen::Index>(row), static_cast<Eigen::Index>(col)) =
        frame.M(static_cast<unsigned int>(row), static_cast<unsigned int>(col));
    }
  }
  // [학습] 출력 포인터가 제공된 경우에만 역참조해서 결과를 씁니다.
  if (joint_array_out != nullptr) {
    *joint_array_out = joints;
  }
  return pose.matrix().allFinite();
}
}  // namespace
```

## §4. 초기화와 하드웨어 인터페이스 요청

**원문 126~163행**

`on_init()`은 파라미터 라이브러리를 준비합니다. `command_interface_configuration()`은 `NONE`, `state_interface_configuration()`은 각 관절의 `position`을 요청합니다.

이 클래스는 하드웨어에 직접 명령을 쓰지 않고 상태를 읽어 토픽을 발행합니다. 상태 인터페이스를 읽는 것과 `/joint_states`를 구독하는 것은 다릅니다. Leader 주요 입력은 전자이고, Follower 피드백에는 후자를 사용합니다.

```cpp

// [학습] 기본 생성자는 비어 있습니다. 실제 ROS 초기 설정은 on_init/on_configure에서 수행합니다.
JointTrajectoryCommandBroadcaster::JointTrajectoryCommandBroadcaster() {}

// [학습] controller manager가 초기화 과정에서 호출하는 생명주기 함수입니다.
controller_interface::CallbackReturn JointTrajectoryCommandBroadcaster::on_init()
{
  try {
    // [학습] 자동 생성된 파라미터 라이브러리의 Listener를 만들고 현재 설정 묶음을 읽습니다.
    param_listener_ = std::make_shared<ParamListener>(get_node());
    params_ = param_listener_->get_params();
  // [학습] 예외를 잡아 ERROR로 보고합니다. SUCCESS/ERROR는 생명주기 호출 결과입니다.
  } catch (const std::exception & e) {
    fprintf(stderr, "Exception thrown during init stage with message: %s \n", e.what());
    return CallbackReturn::ERROR;
  }

  return CallbackReturn::SUCCESS;
}

// [학습] 반환형과 함수 이름이 여러 줄에 걸쳐 있습니다. 다음 const는 객체 상태를 바꾸지 않겠다는 뜻입니다.
controller_interface::InterfaceConfiguration
JointTrajectoryCommandBroadcaster::command_interface_configuration() const
{
  // [학습] NONE: 이 플러그인은 하드웨어 command 인터페이스를 직접 점유하지 않습니다.
  return controller_interface::InterfaceConfiguration{
    controller_interface::interface_configuration_type::NONE};
}

controller_interface::InterfaceConfiguration JointTrajectoryCommandBroadcaster::
state_interface_configuration()
const
{
  controller_interface::InterfaceConfiguration state_interfaces_config;

  // [학습] INDIVIDUAL: 필요한 state 인터페이스 이름을 하나씩 명시합니다.
  state_interfaces_config.type = controller_interface::interface_configuration_type::INDIVIDUAL;
  // [학습] 범위 기반 for문입니다. 각 관절의 position 인터페이스를 요청합니다.
  for (const auto & joint : params_.left_joints) {
    state_interfaces_config.names.push_back(joint + "/" + HW_IF_POSITION);
  }
  for (const auto & joint : params_.right_joints) {
    state_interfaces_config.names.push_back(joint + "/" + HW_IF_POSITION);
  }
  return state_interfaces_config;
}
```

## §5. on_configure(): 설정 검증

**원문 164~209행**

문자열 `teleop_reference_mode`를 enum으로 해석합니다. 상대 모드 이득·가중치와 그리퍼 설정이 적절한지 검사합니다. `params_`의 자료형과 선언 코드는 파라미터 YAML에서 생성되므로 이 `.cpp`에 모든 `declare_parameter` 호출이 보이지 않습니다.

아래 §5~§9는 **같은 on_configure 함수의 연속 구간**입니다. 구간을 나누어 보여줄 뿐 각각 독립 컴파일 가능한 함수는 아닙니다.

```cpp

// [학습] 설정 단계: 토픽·버퍼·초기 follower 상태·기구학을 준비합니다. 아래는 긴 함수 하나입니다.
controller_interface::CallbackReturn JointTrajectoryCommandBroadcaster::on_configure(
  const rclcpp_lifecycle::State & /*previous_state*/)
{
  if (!param_listener_) {
    RCLCPP_ERROR(get_node()->get_logger(), "Error encountered during init");
    return controller_interface::CallbackReturn::ERROR;
  }
  params_ = param_listener_->get_params();

  // [학습] 문자열 설정을 enum으로 변환하고, 변환 실패 시 configure를 중단합니다.
  const auto reference_mode = parse_teleop_reference_mode(params_.teleop_reference_mode);
  if (!reference_mode) {
    RCLCPP_ERROR(
      get_node()->get_logger(),
      "Invalid teleop_reference_mode '%s'; expected 'absolute' or 'relative'",
      params_.teleop_reference_mode.c_str());
    return controller_interface::CallbackReturn::ERROR;
  }
  teleop_reference_mode_ = *reference_mode;
  // [학습] 상대 모드에서는 이득과 가중치가 양수인지 확인합니다.
  if (teleop_reference_mode_ == TeleopReferenceMode::RELATIVE &&
    (params_.relative_kp_position <= 0.0 || params_.relative_kp_orientation <= 0.0 ||
    params_.relative_weight_position <= 0.0 || params_.relative_weight_orientation <= 0.0 ||
    params_.relative_damping_weight <= 0.0))
  {
    RCLCPP_ERROR(
      get_node()->get_logger(),
      "Relative-pose gains, weights, and damping must all be positive");
    return controller_interface::CallbackReturn::ERROR;
  }
  // [학습] 그리퍼 데드밴드·배율은 유한한 숫자이고 허용 범위에 있어야 합니다.
  if (!std::isfinite(params_.gripper_clutch_deadband) ||
    !std::isfinite(params_.gripper_clutch_scale) ||
    params_.gripper_clutch_deadband < 0.0 || params_.gripper_clutch_scale <= 0.0)
  {
    RCLCPP_ERROR(
      get_node()->get_logger(),
      "gripper_clutch_deadband must be non-negative and gripper_clutch_scale must be positive");
    return controller_interface::CallbackReturn::ERROR;
  }
  RCLCPP_INFO(
    get_node()->get_logger(), "Teleop reference mode: %s",
    teleop_reference_mode_ == TeleopReferenceMode::RELATIVE ? "relative" : "absolute");

  // Map interface if needed
  map_interface_to_joint_state_.clear();
  map_interface_to_joint_state_[HW_IF_POSITION] = params_.map_interface_to_joint_state.position;

```

## §6. 그룹별 publisher·버퍼·저장 자세 준비

**원문 210~336행**

왼쪽/오른쪽을 `groups`로 묶어 같은 코드를 재사용합니다. `group_joints`에 관절 목록을, 다른 map에 오프셋·반전 목록·publisher·모드를 저장합니다.

토픽 문자열이 상대 이름이므로 실제 토픽은 launch의 namespace/remap에 따라 달라집니다. `trajectory_groups_ = groups`는 관절이 비어 있는 그룹까지 이름을 넣으므로 설정을 읽을 때 이 점도 확인합니다.

`left_save_pose_3` 같은 이름은 문자열 결합으로 동적으로 만듭니다. 저장 자세 ID는 `uint8_t`로 바뀌므로 설정할 때 0~255 표현 범위를 고려해야 합니다.

```cpp
  // [학습] try 블록 안에서 ROS 객체나 컨테이너 구성 중 발생한 예외를 아래 catch에서 처리합니다.
  try {
    // Create publishers for left and right groups
    std::vector<std::string> groups = {"left", "right"};

    // [학습] left와 right를 같은 코드로 처리합니다. group_name 값만 달라집니다.
    for (const auto & group_name : groups) {
      // Get joints for this group
      std::vector<std::string> group_joints;
      if (group_name == "left" && !params_.left_joints.empty()) {
        group_joints = params_.left_joints;
      } else if (group_name == "right" && !params_.right_joints.empty()) {
        group_joints = params_.right_joints;
      }

      // [학습] 이 그룹에 관절이 없으면 현재 반복을 건너뜁니다. 단, 아래 trajectory_groups_에는 두 이름을 넣습니다.
      if (group_joints.empty()) {
        continue;  // Skip empty groups
      }

      // [학습] map[group_name]에 관절 이름 vector를 저장합니다.
      group_joint_names_[group_name] = group_joints;

      // [학습] offset은 기준점 차이 보정, reverse 목록은 회전 방향 부호 반전용입니다.
      // Get offsets for this group
      if (group_name == "left" && !params_.left_offsets.empty()) {
        group_joint_offsets_[group_name] = params_.left_offsets;
      } else if (group_name == "right" && !params_.right_offsets.empty()) {
        group_joint_offsets_[group_name] = params_.right_offsets;
      } else {
        // Initialize empty offsets if not provided
        group_joint_offsets_[group_name] = std::vector<double>();
      }

      // Get reverse joints for this group
      if (group_name == "left" && !params_.left_reverse_joints.empty()) {
        group_reverse_joints_[group_name] = params_.left_reverse_joints;
      } else if (group_name == "right" && !params_.right_reverse_joints.empty()) {
        group_reverse_joints_[group_name] = params_.right_reverse_joints;
      } else {
        // Initialize empty reverse joints if not provided
        group_reverse_joints_[group_name] = std::vector<std::string>();
      }


      // [학습] 슬래시로 시작하지 않는 상대 토픽 이름입니다. 최종 절대 이름은 namespace와 remap에 달려 있습니다.
      // Create topic name with group-specific namespace
      std::string topic_name;
      topic_name = "joint_trajectory_command_broadcaster_" + group_name + "/joint_trajectory";
      group_topic_names_[group_name] = topic_name;

      // Create publisher for this group
      // [학습] create_publisher<메시지타입>: 이 타입의 메시지를 보낼 publisher를 만듭니다.
      joint_trajectory_publishers_[group_name] =
        get_node()->create_publisher<trajectory_msgs::msg::JointTrajectory>(
        topic_name, rclcpp::SystemDefaultsQoS());

      // [학습] 주기 제어 루프에서 발행을 돕는 래퍼입니다. 사용했다고 함수 전체의 실시간성이 보장되지는 않습니다.
      realtime_joint_trajectory_publishers_[group_name] =
        std::make_shared<realtime_tools::RealtimePublisher<trajectory_msgs::msg::JointTrajectory>>(
        joint_trajectory_publishers_[group_name]);

      RCLCPP_INFO(
        get_node()->get_logger(),
        "Created joint trajectory publisher for group '%s' on topic: %s with %zu joints",
        group_name.c_str(), topic_name.c_str(), group_joints.size());

      // [학습] 별도로 JointState 발행기도 만듭니다. 실제 position에 무엇이 담기는지는 update()를 확인하세요.
      // Create joint state publisher for this group with timestamp from update() function
      std::string joint_state_topic_name;
      if (group_name == "left" && !params_.left_joint_states_stamped_topic.empty()) {
        joint_state_topic_name = params_.left_joint_states_stamped_topic;
      } else if (group_name == "right" && !params_.right_joint_states_stamped_topic.empty()) {
        joint_state_topic_name = params_.right_joint_states_stamped_topic;
      } else {
        joint_state_topic_name =
          "joint_trajectory_command_broadcaster_" + group_name + "/joint_states_stamped";
      }
      joint_state_stamped_publishers_[group_name] =
        get_node()->create_publisher<sensor_msgs::msg::JointState>(
        joint_state_topic_name, rclcpp::SystemDefaultsQoS());

      realtime_joint_state_stamped_publishers_[group_name] =
        std::make_shared<realtime_tools::RealtimePublisher<sensor_msgs::msg::JointState>>(
        joint_state_stamped_publishers_[group_name]);

      RCLCPP_INFO(
        get_node()->get_logger(),
        "Created joint state stamped publisher for group '%s' on topic: %s",
        group_name.c_str(), joint_state_topic_name.c_str());
    }

    // [학습] groups는 left/right 둘 다 포함합니다. 위에서 빈 그룹을 건너뛴 것과 구별하세요.
    // Store the groups for later use
    trajectory_groups_ = groups;

    // Initialize group runtimes with mode from parameters
    // [학습] 그룹별 현재 모드와 콜백 전달 버퍼를 준비합니다.
    for (const auto & group_name : trajectory_groups_) {
      bool init_enabled = (group_name == "left") ?
        params_.left_enabled_init : params_.right_enabled_init;
      // [학습] 삼항 연산자: enabled면 TELEOP, 아니면 IDLE을 선택합니다.
      group_runtime_[group_name].mode = init_enabled ? Mode::TELEOP : Mode::IDLE;
      group_enable_command_buffers_[group_name] =
        std::make_shared<realtime_tools::RealtimeBuffer<EnableCommand>>(EnableCommand{});
      group_last_enable_command_sequence_[group_name] = 0;
      group_follower_state_buffers_[group_name] =
        std::make_shared<realtime_tools::RealtimeBuffer<std::vector<double>>>(
        std::vector<double>{});
      group_executed_action_buffers_[group_name] =
        std::make_shared<realtime_tools::RealtimeBuffer<std::vector<double>>>(
        std::vector<double>{});
    }
    RCLCPP_INFO(get_node()->get_logger(),
      "Initial mode: left=%s, right=%s",
      params_.left_enabled_init ? "TELEOP" : "IDLE",
      params_.right_enabled_init ? "TELEOP" : "IDLE");

    // Load save poses per group from parameters (left_save_pose_<id> / right_save_pose_<id>)
    // [학습] 저장 자세의 ID마다 left_save_pose_3 같은 파라미터 이름을 조립합니다.
    for (int64_t id : params_.save_pose_ids) {
      for (const auto & group_name : trajectory_groups_) {
        std::string pname = group_name + "_save_pose_" + std::to_string(id);
        if (!get_node()->has_parameter(pname)) {
          get_node()->declare_parameter(pname, std::vector<double>{});
        }
        auto values = get_node()->get_parameter(pname).as_double_array();
        if (!values.empty()) {
          // [학습] 명시적 정수 변환입니다. uint8_t는 0~255 범위이며 여기에는 별도 범위 검사가 없습니다.
          group_save_poses_[group_name][static_cast<uint8_t>(id)] = values;
          RCLCPP_INFO(get_node()->get_logger(),
            "Loaded %s (%zu values)", pname.c_str(), values.size());
        }
      }
    }

    // Follower feedback establishes the initial output and remains available
    // to refresh the idle target after another command source moves the robot.
    for (const auto & group_name : trajectory_groups_) {
      group_last_target_initialized_[group_name] = false;
    }
```

## §7. Follower 상태 콜백: 이름으로 관절 정렬하기

**원문 337~382행**

`make_follower_cb("left")`는 **왼팔용 콜백 함수를 반환**합니다. 그 반환된 함수는 메시지가 도착할 때 실행됩니다.

예: 설정 순서가 `[jointA, jointB]`, 수신 메시지가 `name=[jointB,jointA]`, `position=[0.2,0.1]`이면 저장 배열은 `[0.1,0.2]`가 됩니다. 단순히 메시지 배열을 그대로 복사하면 관절이 뒤바뀔 수 있습니다.

정상 값이 모두 모이면 `writeFromNonRT()`로 버퍼에 기록합니다. 이 콜백 안에서 TELEOP 계산이나 publish를 수행하지 않습니다.

```cpp

    auto make_follower_cb = [this](const std::string & group_name) {
        return [this, group_name](sensor_msgs::msg::JointState::SharedPtr msg) {
                 // [학습] 바깥 람다는 그룹명을 받아, 그 그룹용 메시지 콜백을 만들어 반환하는 함수입니다.
                 const auto & joints = group_joint_names_[group_name];
                 // [학습] 안쪽 람다는 나중에 메시지를 받을 때 호출됩니다. group_name을 값으로 캡처해 유지합니다.
                 if (joints.empty()) {
                   return;
                 }
                 std::vector<double> positions(joints.size(), kUninitializedValue);
                 std::vector<std::string> missing_joints;
                 const size_t n = std::min(msg->name.size(), msg->position.size());
                 for (size_t i = 0; i < joints.size(); ++i) {
                   // [학습] name과 position 중 짧은 길이까지만 접근해 불일치에 대비합니다.
                   for (size_t j = 0; j < n; ++j) {
                     // [학습] 메시지 순서를 믿지 않고 관절 이름으로 찾아, 설정된 그룹 순서로 재배열합니다.
                     if (msg->name[j] == joints[i]) {
                       positions[i] = msg->position[j];
                       break;
                     }
                   }
                   if (!std::isfinite(positions[i])) {
                     missing_joints.push_back(joints[i]);
                   }
                 }
                 // [학습] 찾지 못했거나 비정상 값인 관절을 모아 이 메시지를 사용하지 않도록 합니다.
                 if (!missing_joints.empty()) {
                   RCLCPP_WARN_THROTTLE(
            get_node()->get_logger(), *get_node()->get_clock(), 2000,
            "[%s] waiting for follower joint_states on topic with required joints; "
            "missing %zu/%zu joints, first missing: %s",
            group_name.c_str(), missing_joints.size(), joints.size(),
              missing_joints.front().c_str());
                   return;
                 }
                 auto buffer_it = group_follower_state_buffers_.find(group_name);
                 if (buffer_it == group_follower_state_buffers_.end() || !buffer_it->second) {
                   RCLCPP_ERROR(
                     get_node()->get_logger(), "[%s] follower state buffer is unavailable",
                     group_name.c_str());
                   return;
                 // [학습] 콜백에서는 목표를 바로 변경하지 않고 버퍼에 기록합니다. update 쪽에서 읽습니다.
                 }
                 buffer_it->second->writeFromNonRT(positions);
               };
      };

    // [학습] 각 그룹에 방금 만든 콜백을 연결합니다. 구독 객체를 멤버로 보관해 수명을 유지합니다.
    left_follower_js_sub_ = get_node()->create_subscription<sensor_msgs::msg::JointState>(
      params_.left_follower_joint_states_topic, rclcpp::SystemDefaultsQoS(),
      make_follower_cb("left"));
    right_follower_js_sub_ = get_node()->create_subscription<sensor_msgs::msg::JointState>(
      params_.right_follower_joint_states_topic, rclcpp::SystemDefaultsQoS(),
```

## §8. 실행 명령과 enable 명령 콜백

**원문 383~442행**

Follower 실측값은 실제 위치이고, executed action은 최종 전달된 목표입니다. 물체를 잡은 그리퍼는 접촉으로 목표 위치까지 못 갈 수 있어 둘이 다릅니다. 이 코드는 그 차이를 고려해 마지막 그리퍼 목표를 따로 보존합니다.

`left_command`/`right_command`의 `UInt8`는 모드 변경 명령입니다. 콜백은 `handle_enable_msg()`를 호출해 버퍼에 적고, `update()`가 실제 상태를 바꿉니다.

```cpp
      make_follower_cb("right"));

    // The measured gripper position can deflect under contact. Preserve the
    // command that actually reached the follower so an IDLE rebase does not
    // convert that deflection into a zero-error (and therefore weak) hold.
    // [학습] 실제 측정된 그리퍼 위치와 이전에 전달된 그리퍼 목표를 구별하기 위한 콜백입니다.
    auto make_executed_action_cb = [this](const std::string & group_name) {
        return [this, group_name](trajectory_msgs::msg::JointTrajectory::SharedPtr msg) {
                 const auto & joints = group_joint_names_[group_name];
                 if (msg->joint_names != joints || msg->points.size() != 1 ||
                   // [학습] 최종 명령은 관절 이름과 순서, 점 개수 1, 배열 길이가 정확히 맞아야 받습니다.
                   msg->points.front().positions.size() != joints.size())
                 {
                   RCLCPP_WARN_THROTTLE(
                     get_node()->get_logger(), *get_node()->get_clock(), 2000,
                     "[%s] Ignoring malformed final executed action",
                     group_name.c_str());
                   return;
                 }
                 std::vector<double> positions;
                 if (!try_store_finite_action(msg->points.front().positions, positions)) {
                   RCLCPP_WARN_THROTTLE(
                     // [학습] 유한한 값으로 이루어진 완전한 목표인지 보조 함수로 확인합니다.
                     get_node()->get_logger(), *get_node()->get_clock(), 2000,
                     "[%s] Ignoring non-finite final executed action",
                     group_name.c_str());
                   return;
                 }
                 auto buffer_it = group_executed_action_buffers_.find(group_name);
                 if (buffer_it == group_executed_action_buffers_.end() || !buffer_it->second) {
                   RCLCPP_ERROR(
                     get_node()->get_logger(), "[%s] executed action buffer is unavailable",
                     group_name.c_str());
                   return;
                 }
                 buffer_it->second->writeFromNonRT(positions);
               };
      };

    left_executed_action_sub_ =
      get_node()->create_subscription<trajectory_msgs::msg::JointTrajectory>(
      params_.left_executed_action_topic, rclcpp::QoS(1).reliable().transient_local(),
      // [학습] transient_local: 발행자가 보관한 최근 값을 늦게 연결한 구독자도 받을 수 있도록 요청합니다.
      make_executed_action_cb("left"));
    right_executed_action_sub_ =
      get_node()->create_subscription<trajectory_msgs::msg::JointTrajectory>(
      params_.right_executed_action_topic, rclcpp::QoS(1).reliable().transient_local(),
      make_executed_action_cb("right"));

    // Enable topic subscriptions
    //   0=disable, 1=enable, 2=toggle, 3+=disable + trigger save pose <N>
    // [학습] UInt8 명령 수신 콜백입니다. 이 자리에서는 버퍼에 기록하며 실제 모드 적용은 update에서 합니다.
    left_enable_sub_ = get_node()->create_subscription<std_msgs::msg::UInt8>(
      "/leader/left_command", rclcpp::SystemDefaultsQoS(),
      [this](std_msgs::msg::UInt8::SharedPtr msg) {
        handle_enable_msg("left", msg->data);
      });

    right_enable_sub_ = get_node()->create_subscription<std_msgs::msg::UInt8>(
      "/leader/right_command", rclcpp::SystemDefaultsQoS(),
      [this](const std_msgs::msg::UInt8::SharedPtr msg) {
        handle_enable_msg("right", msg->data);
      });

    // Block until follower joint_states init last_target for every group
```

## §9. 초기 피드백·URDF 대기와 관절 한계

**원문 443~555행**

configure는 follower 관절 상태와 URDF를 기다립니다. URDF에서 관절 위치·속도 한계를 읽고 상대 모드의 FK/Jacobian 모델을 구성합니다.

`while + sleep_for`는 메시지를 직접 처리하지 않습니다. 구독 콜백을 실행하는 ROS executor 환경이 별도로 필요합니다. 이 대기에는 코드상 timeout이 없습니다.

지역 구독 `urdf_sub`와 지역 bool의 참조 캡처가 나옵니다. 콜백과 지역 변수의 수명 및 동시 접근은 고급 C++에서 검토할 지점입니다. 현재 코드를 읽는 단계에서는 “URDF를 한 번 받고 설정을 마치려는 구조”로 이해하면 됩니다.

```cpp
    // [학습] configure가 follower 상태 초기화를 기다리는 반복입니다. 제한 시간이 없고 콜백 실행 환경이 필요합니다.
    while (rclcpp::ok()) {
      bool all_init = true;
      for (const auto & g : trajectory_groups_) {
        // [학습] 각 그룹에 유효한 follower 상태가 들어오면 초기 목표를 설정합니다.
        if (!group_last_target_initialized_[g] && sync_follower_target(g)) {
          group_last_target_initialized_[g] = true;
          RCLCPP_INFO(
            get_node()->get_logger(),
            "[%s] last_target initialized from follower joint_states using joint-name order",
            g.c_str());
        }
        if (!group_last_target_initialized_[g]) {
          all_init = false;
        }
      }
      if (all_init) {
        break;
      }
      RCLCPP_WARN_THROTTLE(
        get_node()->get_logger(), *get_node()->get_clock(), 2000,
        "Waiting for follower joint_states to initialize last_target...");
      std::this_thread::sleep_for(std::chrono::milliseconds(100));
    }
// [학습] 100ms 동안 현재 스레드를 쉽니다. 이 sleep 자체가 ROS 콜백을 실행해 주지는 않습니다.

    RCLCPP_INFO(get_node()->get_logger(), "Controller configured successfully.");
  } catch (const std::exception & e) {
    // get_node() may throw, logging raw here
    fprintf(stderr, "Exception thrown during init stage with message: %s \n", e.what());
    return CallbackReturn::ERROR;
  }

  // [학습] Leader URDF를 파싱합니다. 실패해도 이 지점에서는 로그만 남기고 계속 진행합니다.
  const std::string & urdf = get_robot_description();
  is_model_loaded_ = !urdf.empty() && model_.initString(urdf);
  if (!is_model_loaded_) {
    RCLCPP_ERROR(
      get_node()->get_logger(),
      "Failed to parse robot description. Will proceed without URDF-based filtering.");
  }

  // Subscribe to follower's robot_description once to extract joint limits for TELEOP clamping
  bool follower_urdf_loaded = false;
  // [학습] 이 구독은 지역 변수입니다. configure가 끝나면 핸들이 해제됩니다.
  auto urdf_sub = get_node()->create_subscription<std_msgs::msg::String>(
    params_.follower_robot_description_topic,
    rclcpp::QoS(1).transient_local().reliable(),
    // [학습] 지역 변수 follower_urdf_loaded를 참조 캡처합니다. 콜백과 대기 루프가 이 상태를 공유합니다.
    [this, &follower_urdf_loaded](std_msgs::msg::String::SharedPtr msg) {
      if (follower_urdf_loaded) {
        return;
      }
      urdf::Model follower_model;
      if (!follower_model.initString(msg->data)) {
        RCLCPP_ERROR(get_node()->get_logger(),
          "Failed to parse follower robot_description");
        return;
      }
      for (const auto & group_name : trajectory_groups_) {
        const auto & joints = group_joint_names_[group_name];
        std::vector<double> lowers, uppers, velocities;
        for (const auto & jn : joints) {
          // [학습] URDF의 관절 한계가 있으면 위치 하한·상한과 속도 한계를 저장합니다.
          auto j = follower_model.getJoint(jn);
          if (j && j->limits) {
            lowers.push_back(j->limits->lower);
            uppers.push_back(j->limits->upper);
            velocities.push_back(j->limits->velocity);
          } else {
            // [학습] 한계 정보를 못 찾으면 무한대를 넣습니다. 이 코드가 임의의 유한 한계를 만들어 주지는 않습니다.
            lowers.push_back(-std::numeric_limits<double>::infinity());
            uppers.push_back(std::numeric_limits<double>::infinity());
            velocities.push_back(std::numeric_limits<double>::infinity());
            RCLCPP_WARN(get_node()->get_logger(),
              "[%s] No limit for joint '%s' in follower URDF",
              group_name.c_str(), jn.c_str());
          }
        }
        group_lower_limits_[group_name] = lowers;
        group_upper_limits_[group_name] = uppers;
        group_velocity_limits_[group_name] = velocities;
        RCLCPP_INFO(get_node()->get_logger(),
          "[%s] Loaded follower joint limits (%zu joints)",
          group_name.c_str(), lowers.size());
      }
      // [학습] 상대 모드일 때 Leader/Follower의 별도 운동학 체인을 구성합니다.
      if (teleop_reference_mode_ == TeleopReferenceMode::RELATIVE &&
        !configure_relative_pose_kinematics(model_, follower_model))
      {
        RCLCPP_ERROR(
          get_node()->get_logger(), "Failed to configure relative-pose kinematics");
      }
      follower_urdf_loaded = true;
    });

  RCLCPP_INFO(get_node()->get_logger(),
    "Waiting for follower robot_description on '%s'...",
    params_.follower_robot_description_topic.c_str());
  while (rclcpp::ok() && !follower_urdf_loaded) {
    RCLCPP_WARN_THROTTLE(
      // [학습] Follower URDF를 받을 때까지 대기합니다. 여기에도 별도 timeout은 없습니다.
      get_node()->get_logger(), *get_node()->get_clock(), 2000,
      "Still waiting for follower robot_description...");
    std::this_thread::sleep_for(std::chrono::milliseconds(100));
  }
  if (!rclcpp::ok()) {
    return CallbackReturn::ERROR;
  }
  // [학습] 상대 모드에서 필요한 그룹별 운동학이 모두 준비됐는지 최종 확인합니다.
  if (teleop_reference_mode_ == TeleopReferenceMode::RELATIVE &&
    group_pose_kinematics_.size() != group_joint_names_.size())
  {
    RCLCPP_ERROR(
      get_node()->get_logger(),
      "Relative-pose kinematics are incomplete (%zu/%zu groups)",
      group_pose_kinematics_.size(), group_joint_names_.size());
    return CallbackReturn::ERROR;
  }
  // urdf_sub goes out of scope here, effectively unsubscribing

  return CallbackReturn::SUCCESS;
}

```

## §10. 활성화·비활성화

**원문 556~636행**

활성화에서는 state 데이터 매핑을 만들고 관절 개수와 오프셋 개수를 맞춥니다. JointState 출력 배열은 여기서 미리 크기를 정합니다.

비활성화에서는 내부 컨테이너를 비웁니다. 일반적으로 “비활성화 후 재활성화”를 고려할 때 어떤 데이터가 configure에서만 만들어지고 어떤 데이터가 activate에서 복원되는지 확인해야 합니다. 여기서는 재활성화 성공을 시험한 것이 아니라 실제 clear 목록을 읽은 것입니다.

```cpp
// [학습] 활성화 단계입니다. 상태 인터페이스와 메시지 저장 공간을 점검합니다.
controller_interface::CallbackReturn JointTrajectoryCommandBroadcaster::on_activate(
  const rclcpp_lifecycle::State & /*previous_state*/)
{
  if (!init_joint_data()) {
    RCLCPP_ERROR(
      get_node()->get_logger(), "None of requested interfaces exist. Controller will not run.");
    return CallbackReturn::ERROR;
  }

  // Check offsets for each group
  for (const auto & group_name : trajectory_groups_) {
    const auto & group_joints = group_joint_names_[group_name];
    const size_t num_joints = group_joints.size();

    // [학습] 오프셋이 비어 있으면 관절 개수만큼 0으로 채웁니다. 주어진 개수가 틀리면 오류입니다.
    if (group_joint_offsets_[group_name].empty()) {
      // If no offsets provided, use zeros
      group_joint_offsets_[group_name].assign(num_joints, 0.0);
    } else if (group_joint_offsets_[group_name].size() != num_joints) {
      RCLCPP_ERROR(
        get_node()->get_logger(),
        "The number of provided offsets (%zu) for group '%s' does not match the number of "
        "joints (%zu).",
        group_joint_offsets_[group_name].size(), group_name.c_str(), num_joints);
      return CallbackReturn::ERROR;
    }

    RCLCPP_INFO(
      get_node()->get_logger(),
      "Group '%s' configured with %zu joints and %zu offsets",
      group_name.c_str(), num_joints, group_joint_offsets_[group_name].size());
  }

  // Pre-allocate joint state message size for each group (real-time safety)
  // Must be done after init_joint_data() so group_joint_names_ is populated
  for (const auto & group_name : trajectory_groups_) {
    // [학습] JointState 메시지 배열을 미리 확보합니다. 모든 update 경로가 무할당이라는 의미는 아닙니다.
    const auto & group_joints = group_joint_names_[group_name];

    auto it = realtime_joint_state_stamped_publishers_.find(group_name);
    if (it != realtime_joint_state_stamped_publishers_.end() && it->second) {
      auto & msg = it->second->msg_;
      const size_t num_joints = group_joints.size();
      msg.name.resize(num_joints);
      msg.position.resize(num_joints, std::numeric_limits<double>::quiet_NaN());
      // [학습] auto &로 publisher 내부 메시지 원본에 접근합니다. resize는 배열 크기를 설정합니다.
      msg.velocity.resize(num_joints, std::numeric_limits<double>::quiet_NaN());
      msg.effort.resize(num_joints, std::numeric_limits<double>::quiet_NaN());

      // Set joint names once (done in on_activate, not in update loop)
      for (size_t i = 0; i < num_joints; ++i) {
        msg.name[i] = group_joints[i];
      }

      RCLCPP_INFO(
        get_node()->get_logger(),
        "Pre-allocated joint state message for group '%s' with %zu joints",
        group_name.c_str(), num_joints);
    }
  }

  return CallbackReturn::SUCCESS;
}


// [학습] 비활성화 시 여러 컨테이너를 비웁니다. 재활성화 시 어떤 구성을 다시 채워야 할지도 살펴볼 지점입니다.
controller_interface::CallbackReturn JointTrajectoryCommandBroadcaster::on_deactivate(
  const rclcpp_lifecycle::State & /*previous_state*/)
{
  joint_names_.clear();
  name_if_value_mapping_.clear();
  group_joint_names_.clear();
  group_joint_offsets_.clear();
  group_topic_names_.clear();
  group_reverse_joints_.clear();
  group_lower_limits_.clear();
  group_upper_limits_.clear();
  group_velocity_limits_.clear();
  group_pose_kinematics_.clear();
  group_last_target_.clear();
  group_follower_state_buffers_.clear();
  group_executed_action_buffers_.clear();
  group_runtime_.clear();

  return CallbackReturn::SUCCESS;
```

## §11. map 보조 함수와 초기 데이터 구성

**원문 637~686행**

`has_any_key()`는 map에 필요한 인터페이스가 있는지 검사하는 함수 템플릿입니다. `init_joint_data()`는 하드웨어 인터페이스 목록을 중첩 map으로 구성합니다.

`use_urdf_to_filter` 조건은 이 구간에서 `joint_names_`를 채우는 데 적용됩니다. 실제 출력은 뒤에서 `group_joint_names_`를 사용하므로, 이 코드만 보고 모든 출력이 자동으로 URDF 필터링된다고 확대 해석하면 안 됩니다.

```cpp
// [학습] 보조 함수 템플릿입니다. map에 후보 키 중 하나라도 있는지 검사합니다.
}

template<typename T>
bool has_any_key(
  const std::unordered_map<std::string, T> & map, const std::vector<std::string> & keys)
// [학습] map의 원소는 key/value 쌍입니다. first는 키, second는 값입니다.
{
  for (const auto & key_item : map) {
    const auto & key = key_item.first;
    if (std::find(keys.cbegin(), keys.cend(), key) != keys.cend()) {
      return true;
    }
  }
  return false;
}

// [학습] 할당받은 state 인터페이스를 관절별 딕셔너리 구조로 정리합니다.
bool JointTrajectoryCommandBroadcaster::init_joint_data()
{
  joint_names_.clear();
  if (state_interfaces_.empty()) {
    return false;
  }

  // Initialize mapping
  // [학습] crbegin/crend는 읽기 전용 역방향 반복자입니다. 마지막 인터페이스부터 순회합니다.
  for (auto si = state_interfaces_.crbegin(); si != state_interfaces_.crend(); si++) {
    if (name_if_value_mapping_.count(si->get_prefix_name()) == 0) {
      name_if_value_mapping_[si->get_prefix_name()] = {};
    }
    std::string interface_name = si->get_interface_name();
    if (map_interface_to_joint_state_.count(interface_name) > 0) {
      interface_name = map_interface_to_joint_state_[interface_name];
    }
    // [학습] 읽기 전 초기값은 NaN으로 둡니다.
    name_if_value_mapping_[si->get_prefix_name()][interface_name] = kUninitializedValue;
  }

  // Filter out joints without position interface (since we want positions)
  // [학습] position 인터페이스와 URDF 존재 여부를 보고 joint_names_ 목록을 만듭니다.
  for (const auto & name_ifv : name_if_value_mapping_) {
    const auto & interfaces_and_values = name_ifv.second;
    if (has_any_key(interfaces_and_values, {HW_IF_POSITION})) {
      if (
        !params_.use_urdf_to_filter || !is_model_loaded_ ||
        model_.getJoint(name_ifv.first))
      {
        joint_names_.push_back(name_ifv.first);
      }
    }
  }

  return true;
}

```

## §12. get_value(): 관절 값 조회

**원문 687~699행**

`map.at(name)`에서 관절을 찾고 그 안에서 인터페이스를 찾습니다. 인터페이스가 없으면 NaN을 반환하지만, **관절 키 자체가 없으면 at()에서 예외**가 발생합니다. “없는 경우 항상 NaN”이 아니라 실패 위치별 동작이 다릅니다.

```cpp
// [학습] 관절/인터페이스 이름으로 수치를 찾는 보조 함수입니다.
double get_value(
  const std::unordered_map<std::string, std::unordered_map<std::string, double>> & map,
  const std::string & name, const std::string & interface_name)
{
  // [학습] at(name)은 관절 키가 없으면 예외를 던집니다. 아래 NaN 처리는 인터페이스가 없을 때입니다.
  const auto & interfaces_and_values = map.at(name);
  const auto interface_and_value = interfaces_and_values.find(interface_name);
  if (interface_and_value != interfaces_and_values.cend()) {
    return interface_and_value->second;
  } else {
    return kUninitializedValue;
  }
}

```

## §13. update() ① 상태 읽기와 명령 반영

**원문 700~748행**

이제 주기 루프입니다. state 인터페이스 → 중첩 map → 그룹별 모드 명령 확인 순서로 진행됩니다.

`get_optional()`에 값이 없으면 이번 주기에는 map을 갱신하지 않습니다. 따라서 이전 값이 남을 수 있습니다. 이 구간에는 입력 timestamp를 통한 신선도 검사가 없습니다.

`EnableCommand.sequence`가 지난 처리 번호와 다를 때만 새 명령을 반영합니다. 이 버퍼는 이벤트를 모두 저장하는 큐가 아니라 최근 값을 전달하는 구조입니다.

```cpp
// [학습] 주기 처리의 중심입니다. time은 호출 시각, period는 이번 제어 주기 길이입니다.
controller_interface::return_type JointTrajectoryCommandBroadcaster::update(
  const rclcpp::Time & time, const rclcpp::Duration & period)
{
  // Update stored values
  // [학습] Leader 하드웨어 상태 핸들에서 최신 값을 읽어 map에 복사합니다.
  for (const auto & state_interface : state_interfaces_) {
    std::string interface_name = state_interface.get_interface_name();
    if (map_interface_to_joint_state_.count(interface_name) > 0) {
      interface_name = map_interface_to_joint_state_[interface_name];
    }
    // [학습] optional 값이 있을 때만 갱신합니다. 없으면 기존 map 값이 남습니다.
    auto value = state_interface.get_optional();
    if (value) {
      name_if_value_mapping_[state_interface.get_prefix_name()][interface_name] = *value;
    }
  }

  // Publish JointTrajectory messages for each group with current positions
  // [학습] 왼쪽/오른쪽 그룹을 각각 처리합니다. Python의 dict 순회처럼 생각해도 됩니다.
  for (const auto & group_name : trajectory_groups_) {
    const auto & group_joints = group_joint_names_[group_name];

    // Safely get group offsets and reverse joints
    std::vector<double> group_offsets;
    std::vector<std::string> group_reverse_joints;

    auto offsets_it = group_joint_offsets_.find(group_name);
    if (offsets_it != group_joint_offsets_.end()) {
      group_offsets = offsets_it->second;
    }

    auto reverse_it = group_reverse_joints_.find(group_name);
    if (reverse_it != group_reverse_joints_.end()) {
      group_reverse_joints = reverse_it->second;
    }

    const size_t num_joints = group_joints.size();
    auto & last_target = group_last_target_[group_name];
    // [학습] 참조이므로 last_target이나 rt를 수정하면 해당 그룹의 멤버 데이터가 바뀝니다.
    auto & rt = group_runtime_[group_name];

    auto command_buffer_it = group_enable_command_buffers_.find(group_name);
    if (command_buffer_it != group_enable_command_buffers_.end() && command_buffer_it->second) {
      // [학습] 콜백이 넣어 둔 최신 명령을 읽고 일련번호로 중복 처리를 피합니다.
      const auto * command = command_buffer_it->second->readFromRT();
      if (command && command->sequence != 0 &&
        command->sequence != group_last_enable_command_sequence_[group_name])
      {
        group_last_enable_command_sequence_[group_name] = command->sequence;
        apply_enable_msg(group_name, command->data);
      // [학습] 모드 변경은 여기서 적용합니다. ROS 구독 콜백과 update의 역할을 나눈 구조입니다.
      }
    }

    // Update last_target based on mode
```

## §14. update() ② IDLE

**원문 749~762행**

IDLE에서는 follower 상태로 내부 목표를 맞춥니다. 다른 소스가 로봇을 움직인 뒤 TELEOP으로 돌아갈 때 옛 목표가 다시 나오는 것을 줄이려는 동작입니다.

`break`는 switch를 빠져나올 뿐 update 함수나 그룹 반복문 전체를 종료하지 않습니다. 따라서 뒤의 공통 발행부는 계속 실행될 수 있습니다.

```cpp
    // [학습] switch는 현재 모드에 해당하는 case만 실행합니다. case 끝 break는 switch를 빠져나옵니다.
    switch (rt.mode) {
      // [학습] IDLE은 발행 중단이 아닙니다. follower 상태로 목표를 맞춘 후 아래 공통 발행부로 갑니다.
      case Mode::IDLE: {
        // WAYPOINT/POLICY can move the physical follower while the leader is
        // idle. Keep the pending TELEOP sample at the measured pose so route
        // selection cannot replay the pre-handoff command.
          // [학습] 실패하면 기존 유효 목표를 유지하고 경고합니다.
          if (!sync_follower_target(group_name)) {
            RCLCPP_WARN_THROTTLE(
              get_node()->get_logger(), *get_node()->get_clock(), 2000,
              "[%s] No valid follower feedback; holding the last finite action",
              group_name.c_str());
          }
          break;
        }

```

## §15. update() ③ TELEOP

**원문 763~873행**

Leader 위치를 읽어 방향 반전과 오프셋을 적용한 `leader_reference`를 만듭니다. 모든 값이 유효할 때 상대 모드 또는 절대 모드로 `candidate`를 계산합니다.

절대 모드의 팔 관절은 Leader 기준을 따르되 진입 시 smoothstep 보간을 적용합니다. 상대 모드는 별도 FK/IK 함수로 계산합니다. 그리퍼는 두 모드 모두 기준점을 잡고 데드밴드를 넘은 움직임을 반영합니다.

최종 배열이 모두 유효할 때만 `last_target`을 교체합니다. 오류 시 로그를 남기고 기존 목표를 유지하는 경로가 많으므로 `break`가 무엇을 끝내는지 살펴보세요.

```cpp
      // [학습] TELEOP: Leader 위치에서 목표 후보를 만들고 유효할 때만 last_target을 교체합니다.
      case Mode::TELEOP: {
          const auto & lowers = group_lower_limits_[group_name];
          const auto & uppers = group_upper_limits_[group_name];
          const bool clamp_enabled =
            lowers.size() == num_joints && uppers.size() == num_joints;

          std::vector<double> leader_reference(num_joints, kUninitializedValue);
          for (size_t i = 0; i < num_joints; ++i) {
            // [학습] 우선 Leader 입력 전체를 임시 배열에 모읍니다. 일부만 유효한 목표가 섞이지 않게 합니다.
            double leader_val =
              get_value(name_if_value_mapping_, group_joints[i], HW_IF_POSITION);
            if (!std::isfinite(leader_val)) {
              break;
            // [학습] 목록에 있는 관절은 부호를 반전합니다. 이후 오프셋을 더합니다.
            }
            if (std::find(group_reverse_joints.begin(), group_reverse_joints.end(),
              group_joints[i]) != group_reverse_joints.end())
            {
              leader_val = -leader_val;
            }
            if (i < group_offsets.size()) {
              leader_val += group_offsets[i];
            }
            leader_reference[i] = leader_val;
          }

          // [학습] all_of와 람다로 모든 원소가 유한한 값인지 검사합니다.
          if (!std::all_of(
              leader_reference.begin(), leader_reference.end(), [](const double value) {
                return std::isfinite(value);
              }))
          {
            RCLCPP_ERROR_THROTTLE(
              get_node()->get_logger(), *get_node()->get_clock(), 2000,
              "[%s] Invalid leader state; holding the last finite action",
              group_name.c_str());
            break;
          }

          // [학습] TELEOP 시작 시 그리퍼 기준점을 한 번 캡처합니다. 움직이지 않은 그리퍼가 튀지 않게 하기 위함입니다.
          if (!rt.gripper_clutch.anchored) {
            if (last_target.size() != num_joints ||
              !std::all_of(last_target.begin(), last_target.end(), [](const double value) {
                return std::isfinite(value);
              }))
            {
              RCLCPP_ERROR_THROTTLE(
                get_node()->get_logger(), *get_node()->get_clock(), 2000,
                "[%s] Cannot capture gripper clutch without a finite held action",
                group_name.c_str());
              break;
            }
            rt.gripper_clutch.leader_anchor = leader_reference;
            rt.gripper_clutch.command_anchor = last_target;
            rt.gripper_clutch.anchored = true;
          }

          // [학습] candidate는 임시 결과입니다. 계산 중 오류가 나면 기존 목표를 보존합니다.
          std::vector<double> candidate(num_joints, kUninitializedValue);
          // [학습] 상대 모드: FK/Jacobian을 이용하는 별도 함수로 목표를 계산합니다.
          if (teleop_reference_mode_ == TeleopReferenceMode::RELATIVE) {
            if (!update_relative_pose_target(
                group_name, leader_reference, period.seconds(), candidate))
            {
              RCLCPP_ERROR_THROTTLE(
                get_node()->get_logger(), *get_node()->get_clock(), 2000,
                "[%s] Relative-pose FK/IK failed; holding last action",
                group_name.c_str());
              break;
            }
          } else {
            double blend_alpha = 1.0;
            if (rt.blend.active) {
              // [학습] 경과시간/전체시간을 0~1로 제한해 보간 진행률을 만듭니다.
              const double elapsed = (get_node()->now() - rt.blend.start_time).seconds();
              const double t =
                std::clamp(elapsed / params_.leader_blend_duration, 0.0, 1.0);
              // [학습] 3t²-2t³는 양 끝에서 기울기가 0인 cubic smoothstep입니다.
              blend_alpha = 3.0 * t * t - 2.0 * t * t * t; // cubic smoothstep
              if (t >= 1.0) {
                rt.blend.active = false;
              }
            }
            // [학습] 절대 모드에서도 그리퍼는 팔 관절과 다르게 클러치 규칙을 적용합니다.
            for (size_t i = 0; i < num_joints; ++i) {
              if (is_gripper_joint(group_joints[i])) {
                if (!try_map_gripper_clutch_target(
                    leader_reference[i], rt.gripper_clutch.leader_anchor[i],
                    rt.gripper_clutch.command_anchor[i], params_.gripper_clutch_deadband,
                    params_.gripper_clutch_scale, candidate[i]))
                {
                  candidate[i] = kUninitializedValue;
                  continue;
                }
                if (clamp_enabled) {
                  candidate[i] = std::clamp(candidate[i], lowers[i], uppers[i]);
                }
                continue;
              }
              double leader_val = leader_reference[i];
              // [학습] 절대 모드 팔 관절: 진입 중에는 기존 목표와 Leader 값을 섞고, 이후 Leader 값을 사용합니다.
              if (rt.blend.active && i < rt.blend.start_pos.size()) {
                leader_val = (1.0 - blend_alpha) * rt.blend.start_pos[i] +
                  blend_alpha * leader_val;
              }
              // [학습] Follower URDF에서 읽은 위치 한계로 목표를 제한합니다.
              if (clamp_enabled) {
                leader_val = std::clamp(leader_val, lowers[i], uppers[i]);
              }
              candidate[i] = leader_val;
            }
          }

          if (!try_store_finite_action(candidate, last_target)) {
            // [학습] 전체 candidate가 유효한 경우에만 last_target을 교체합니다.
            RCLCPP_ERROR_THROTTLE(
            get_node()->get_logger(), *get_node()->get_clock(), 2000,
            "[%s] Invalid leader state; holding the last finite action",
            group_name.c_str());
          }
          break;
        }

```

## §16. update() ④ SAVE_POSE

**원문 874~897행**

저장 자세는 시간 기반 보간입니다. `s=3t²−2t³`, `q=q_start+s(q_target−q_start)`를 매 주기 계산합니다.

이 파일의 저장 자세 완료는 예정 시간이 끝났는지로 판단합니다. 앞서 본 Python `joint_trajectory_executor`의 실제 위치·속도 오차 검사와 다릅니다. 보간이 끝나도 `Mode::SAVE_POSE`에 남아 목표를 유지합니다.

```cpp
      // [학습] 저장 자세 모드: 매 주기 시간 기반 보간값을 계산합니다.
      case Mode::SAVE_POSE: {
        // Cubic interp to target; hold at target after completion
          if (rt.interp.active) {
            double elapsed = (get_node()->now() - rt.interp.start_time).seconds();
            // [학습] 여기 t는 실제 시간값이 아니라 0~1 정규화 진행률입니다.
            double t = std::clamp(elapsed / rt.interp.duration_sec, 0.0, 1.0);
            double s = 3.0 * t * t - 2.0 * t * t * t;
            // [학습] 예정 시간이 끝나면 보간만 종료합니다. 모드는 SAVE_POSE에 남아 마지막 목표를 유지합니다.
            if (t >= 1.0) {
              rt.interp.active = false;
            }

            last_target.resize(num_joints);
            for (size_t i = 0; i < num_joints; ++i) {
              if (i < rt.interp.start_pos.size() && i < rt.interp.target_pos.size()) {
                last_target[i] = rt.interp.start_pos[i] +
                  // [학습] 출발값 + 진행계수 × (도착값-출발값) 형태의 보간입니다.
                  s * (rt.interp.target_pos[i] - rt.interp.start_pos[i]);
              }
            }
          }
        // else: hold last_target (which should equal target after completion)
          break;
        }
    }

    // last_target is valid only after first enable/interp
```

### 계산 실습 1 — 저장 자세 보간

원문은 C++이지만 숫자 변화만 Python으로 재현합니다. 관절 하나를 0에서 1 rad까지 옮기는 예입니다. `t`는 정규화된 진행률이며, `s`는 시작과 끝에서 변화율이 완만해지는 계수입니다. 실제 로봇이 이 값에 도착했는지는 이 계산만으로 알 수 없습니다.

In [1]:
for t in (0.0, 0.25, 0.5, 0.75, 1.0):
    s = 3*t*t - 2*t*t*t
    q_start, q_target = 0.0, 1.0
    q = q_start + s * (q_target - q_start)
    print(f"진행률={t:.2f}, 보간계수={s:.5f}, 목표={q:.5f} rad")

진행률=0.00, 보간계수=0.00000, 목표=0.00000 rad
진행률=0.25, 보간계수=0.15625, 목표=0.15625 rad
진행률=0.50, 보간계수=0.50000, 목표=0.50000 rad
진행률=0.75, 보간계수=0.84375, 목표=0.84375 rad
진행률=1.00, 보간계수=1.00000, 목표=1.00000 rad


## §17. update() ⑤ JointTrajectory 발행

**원문 898~916행**

세 모드가 공통으로 거치는 출력 구간입니다. `last_target` 크기와 수치를 확인한 뒤, 위치만 채운 점 하나를 발행합니다.

`header.stamp=0`, `time_from_start=0`으로 구성하며 `period`를 자동으로 점의 시간에 넣지 않습니다. 수신 제어기의 구체적인 시간 해석은 그 제어기 구현에 달려 있습니다. 여기서 확인되는 사실은 “현재 목표를 한 점 메시지로 반복 전송”한다는 것입니다.

```cpp
    // [학습] 아래 발행부는 모든 모드에 공통입니다. IDLE에서도 valid이면 실행됩니다.
    const bool valid = last_target.size() == num_joints &&
      std::all_of(last_target.begin(), last_target.end(), [](double value) {
          return std::isfinite(value);
      });
// [학습] 크기와 모든 수치의 유한성을 확인합니다.

    // Publish trajectory (always when last_target valid)
    auto & realtime_publisher = realtime_joint_trajectory_publishers_[group_name];
    if (valid && realtime_publisher) {
      trajectory_msgs::msg::JointTrajectory traj_msg;
      // [학습] header.stamp가 0이라는 것은 실제 센서 시각을 여기에 기록하지 않는다는 뜻입니다.
      traj_msg.header.stamp = rclcpp::Time(0, 0);
      traj_msg.joint_names = group_joints;
      // [학습] 궤적 점은 하나입니다. 긴 경로 전체가 아니라 현재 계산한 목표를 반복 전달합니다.
      traj_msg.points.resize(1);
      traj_msg.points[0].positions = last_target;
      traj_msg.points[0].time_from_start = rclcpp::Duration(0, 0);
      // [학습] try_publish는 발행 시도입니다. 호출 자체가 최종 제어기의 실행 완료를 의미하지 않습니다.
      realtime_publisher->try_publish(traj_msg);
    }

    // Publish joint state for this group with timestamp from update() function
    // Real-time safe: message size pre-allocated in on_activate(), only copy values here
```

## §18. update() ⑥ JointState 발행과 실제 데이터 의미

**원문 917~981행**

코멘트에는 state 복사나 같은 변환이라는 설명이 있지만, 실제 `msg.position[i]`의 원천은 **last_target**입니다. 즉 이 메시지의 position을 그대로 Leader 원시 센서 측정값이라고 부르면 부정확합니다.

`header.stamp`는 update에 전달된 time입니다. 하드웨어의 실제 샘플 획득 시각과 동일하다고 이 코드만으로 증명할 수는 없습니다. state 인터페이스는 position만 요청하므로 velocity/effort는 일반적으로 NaN 경로에 들어갑니다.

`trylock()`은 기다리지 않고 잠금을 시도합니다. 실패하면 이 주기의 해당 발행을 건너뜁니다.

```cpp
    // [학습] 원래 주석보다 대입문을 우선 보세요. 아래 position에는 raw Leader 값 대신 last_target이 들어갑니다.
    // Apply same transformations (reverse, offsets) as joint_trajectory
    auto joint_state_pub_it = realtime_joint_state_stamped_publishers_.find(group_name);
    if (joint_state_pub_it != realtime_joint_state_stamped_publishers_.end() &&
      joint_state_pub_it->second)
    {
      auto & realtime_joint_state_publisher = joint_state_pub_it->second;
      // [학습] 잠금을 즉시 얻지 못하면 이 주기의 JointState 발행을 건너뜁니다.
      if (realtime_joint_state_publisher->trylock()) {
        auto & msg = realtime_joint_state_publisher->msg_;

        // Set timestamp from update() function argument (actual sensor read time)
        // [학습] JointState에는 update 인자의 time을 기록합니다. 이것이 센서 자체의 timestamp임을 보장하진 않습니다.
        msg.header.stamp = time;
        msg.header.frame_id = "";

        // Copy values from name_if_value_mapping_ to pre-allocated message
        // Apply same reverse and offset transformations as joint_trajectory
        // No string operations, no dynamic allocation - only value copying
        for (size_t i = 0; i < group_joints.size() && i < msg.name.size(); ++i) {
          const std::string & joint_name = group_joints[i];
          // [학습] 이 위치 배열은 계산된 명령 목표입니다. 실제 측정 위치나 순수 Leader 위치로 단정하지 마세요.
          double pos_value = (i < last_target.size()) ? last_target[i] :
            std::numeric_limits<double>::quiet_NaN();

          msg.position[i] = std::isnan(pos_value) ? std::numeric_limits<double>::quiet_NaN() :
            pos_value;

          // Get velocity if available (no reverse/offset for velocity)
          if (name_if_value_mapping_.count(joint_name) > 0) {
            // [학습] 현재 state_interface_configuration은 position만 요청하므로 velocity/effort는 보통 준비되지 않습니다.
            const auto & interfaces = name_if_value_mapping_.at(joint_name);
            auto vel_it = interfaces.find(HW_IF_VELOCITY);
            if (vel_it != interfaces.end() && !std::isnan(vel_it->second)) {
              // Apply reverse sign to velocity if joint is reversed
              double vel_value = vel_it->second;
              if (
                std::find(
                  group_reverse_joints.begin(),
                  group_reverse_joints.end(),
                  joint_name) != group_reverse_joints.end())
              {
                vel_value = -vel_value;
              }
              msg.velocity[i] = vel_value;
            } else {
              msg.velocity[i] = std::numeric_limits<double>::quiet_NaN();
            }

            // Get effort if available (no reverse/offset for effort)
            auto eff_it = interfaces.find(HW_IF_EFFORT);
            if (eff_it != interfaces.end() && !std::isnan(eff_it->second)) {
              msg.effort[i] = eff_it->second;
            } else {
              msg.effort[i] = std::numeric_limits<double>::quiet_NaN();
            }
          } else {
            msg.velocity[i] = std::numeric_limits<double>::quiet_NaN();
            msg.effort[i] = std::numeric_limits<double>::quiet_NaN();
          }
        }

        realtime_joint_state_publisher->unlockAndPublish();
      }
    // [학습] publisher 내부 메시지를 잠금 해제하고 발행합니다.
    }
  }

  return controller_interface::return_type::OK;
}

```

## §19. handle_enable_msg(): 콜백에서 명령 저장

**원문 982~998행**

명령을 받으면 원자적 카운터로 번호를 붙여 버퍼에 기록합니다. `fetch_add(1)`은 증가 전 값을 반환하므로 뒤에 `+1`을 붙여 1부터 시작합니다.

`std::memory_order_relaxed`는 이 카운터의 원자성에 관한 설정입니다. 이것만으로 클래스 전체의 동시 접근 안전성이 생기는 것은 아닙니다. 상태 변경은 다음 update에서 수행합니다.

```cpp
// [학습] 구독 콜백 경로: 명령을 실시간 전달 버퍼에 기록하는 역할입니다.
void JointTrajectoryCommandBroadcaster::handle_enable_msg(
  const std::string & group_name, uint8_t data)
{
  auto buffer_it = group_enable_command_buffers_.find(group_name);
  if (buffer_it == group_enable_command_buffers_.end() || !buffer_it->second) {
    RCLCPP_ERROR(
      get_node()->get_logger(), "[%s] Cannot queue enable command: unknown group",
      group_name.c_str());
    return;
  }

  EnableCommand command;
  // [학습] 원자적 증가로 명령마다 번호를 붙입니다. relaxed는 이 카운터 연산의 원자성만 요구합니다.
  command.sequence = next_enable_command_sequence_.fetch_add(1, std::memory_order_relaxed) + 1;
  command.data = data;
  // [학습] 최근 값 전달 버퍼입니다. update 사이에 여러 명령이 오면 모든 중간 명령 보존을 보장하지 않습니다.
  buffer_it->second->writeFromNonRT(command);
}

```

## §20. apply_enable_msg(): 실제 상태 전환

**원문 999~1105행**

0=IDLE, 1=TELEOP, 2=토글, 그 외=저장 자세 ID입니다. 왼팔·오른팔은 각각 별도 상태를 가집니다.

같은 토글이 짧은 시간 안에 두 번 들어오면 꺼졌다 켜지는 것을 막기 위해 디바운스를 적용합니다. 같은 저장 자세 ID가 반복되면 보간을 다시 시작하지 않습니다.

TELEOP을 끌 때와 켤 때 상대 기준·그리퍼 기준을 초기화합니다. 명령 0의 `sync_follower_target(..., false)`는 그리퍼까지 실측으로 재설정하도록 요청합니다. 따라서 “언제나 그리퍼 명령 보존”이라고 단순화하면 안 됩니다.

```cpp
// [학습] 주기 루프에서 실제 모드와 보간 상태를 바꾸는 함수입니다.
void JointTrajectoryCommandBroadcaster::apply_enable_msg(
  const std::string & group_name, uint8_t data)
{
  auto & rt = group_runtime_[group_name];

  switch (data) {
    // [학습] 0: 추종 중단. 보간을 끄고 follower 측정 상태로 기준을 다시 맞춥니다.
    case 0: // stop
      rt.toggle_gate.reset();
      rt.mode = Mode::IDLE;
      rt.blend.active = false;
      rt.interp.active = false;
      rt.active_save_pose_id = 0;
      // Route setup sends command=0 before TELEOP is enabled. Rebase the
      // complete pending action here, including the gripper, so an executed
      // action left by an earlier session cannot close the gripper at startup.
      // [학습] false를 전달하므로 여기서는 그리퍼까지 측정값으로 다시 맞추도록 요청합니다.
      sync_follower_target(group_name, false);
      reset_relative_pose_anchor(group_name);
      reset_gripper_clutch(group_name);
      RCLCPP_INFO(
        get_node()->get_logger(), "[%s] Teleop stopped; synchronizing with follower feedback",
        group_name.c_str());
      return;

    // [학습] 1: TELEOP 진입. 이미 TELEOP이면 기준점 재캡처를 반복하지 않습니다.
    case 1: // teleop
      rt.toggle_gate.reset();
      if (rt.mode != Mode::TELEOP) {
        // Relative TELEOP starts from the newest measured arm pose, but its
        // gripper anchor must remain the most recently executed command. A
        // loaded gripper's measured deflection is not the commanded grasp.
        // [학습] 상대 모드 진입에서는 마지막으로 명령한 그리퍼 목표를 유지하려고 합니다.
        const bool preserve_commanded_grippers =
          teleop_reference_mode_ == TeleopReferenceMode::RELATIVE;
        sync_follower_target(group_name, preserve_commanded_grippers);
        prepare_teleop_entry(group_name);
      }
      rt.mode = Mode::TELEOP;
      rt.interp.active = false;
      rt.active_save_pose_id = 0;
      RCLCPP_INFO(get_node()->get_logger(), "[%s] Teleop enabled", group_name.c_str());
      return;

    // [학습] 2: 현재 모드를 반전합니다. 너무 빠른 중복 토글은 gate에서 무시합니다.
    case 2:  // toggle
      // [학습] 디바운스: 한 번 누른 이벤트가 중복 전달되어 바로 꺼지는 상황을 줄입니다.
      if (!rt.toggle_gate.accept(
          get_node()->now().seconds(), params_.toggle_debounce_duration_sec))
      {
        RCLCPP_WARN_THROTTLE(
          get_node()->get_logger(), *get_node()->get_clock(), 2000,
          "[%s] Ignoring duplicate toggle inside %.3fs debounce window",
          group_name.c_str(), params_.toggle_debounce_duration_sec);
        return;
      }
      if (rt.mode == Mode::TELEOP) {
        rt.mode = Mode::IDLE;
        rt.blend.active = false;
        rt.active_save_pose_id = 0;
        sync_follower_target(group_name, false);
        reset_relative_pose_anchor(group_name);
        reset_gripper_clutch(group_name);
        RCLCPP_INFO(
          get_node()->get_logger(),
          "[%s] Teleop toggled off; synchronizing with follower feedback",
          group_name.c_str());
      } else {
        const bool preserve_commanded_grippers =
          teleop_reference_mode_ == TeleopReferenceMode::RELATIVE;
        sync_follower_target(group_name, preserve_commanded_grippers);
        prepare_teleop_entry(group_name);
        rt.mode = Mode::TELEOP;
        rt.interp.active = false;
        rt.active_save_pose_id = 0;
        RCLCPP_INFO(get_node()->get_logger(), "[%s] Teleop toggled on", group_name.c_str());
      }
      return;

    // [학습] 그 외 uint8 값은 저장 자세 ID로 해석합니다. 설정된 ID만 실행할 수 있습니다.
    default: {  // 3, 4, ... : save pose N
        rt.toggle_gate.reset();
        // [학습] 같은 저장 자세 명령을 반복 받아도 보간 시간을 매번 처음부터 다시 시작하지 않습니다.
        if (rt.mode == Mode::SAVE_POSE && rt.active_save_pose_id == data) {
          RCLCPP_WARN_THROTTLE(
          get_node()->get_logger(), *get_node()->get_clock(), 2000,
          "[%s] Ignoring repeated save-pose trigger id=%u",
          group_name.c_str(), data);
          return;
        }
        auto gp_it = group_save_poses_.find(group_name);
        if (gp_it == group_save_poses_.end()) {
          return;
        }
        auto pose_it = gp_it->second.find(data);
        if (pose_it == gp_it->second.end()) {
          RCLCPP_WARN(get_node()->get_logger(),
          "[%s] No save pose defined for id=%u", group_name.c_str(), data);
          return;
        }
        // [학습] 저장 자세 배열의 크기와 유효성을 확인하고 보간 시작 상태를 준비합니다.
        if (!start_save_pose_interp(group_name, pose_it->second)) {
          return;
        }
        rt.mode = Mode::SAVE_POSE;
        rt.blend.active = false;
        rt.active_save_pose_id = data;
        reset_relative_pose_anchor(group_name);
        reset_gripper_clutch(group_name);
        RCLCPP_INFO(get_node()->get_logger(),
        "[%s] Start interpolation to save pose %u", group_name.c_str(), data);
        return;
      }
  }
}

```

### 계산 실습 2 — 그룹별 상태를 기억하는 방식

아래는 원문의 모든 검사를 복제한 구현이 아니라 **switch와 상태 변수 개념만** 보여 주는 단순 모델입니다. 디바운스·실측 동기화·버퍼는 생략했습니다. 실제 저장 자세 ID는 파라미터에 등록되어 있어야 합니다.

In [2]:
mode = "IDLE"
saved_ids = {3, 4}
for command in [1, 2, 2, 3, 0]:
    if command == 0:
        mode = "IDLE"
    elif command == 1:
        mode = "TELEOP"
    elif command == 2:
        mode = "IDLE" if mode == "TELEOP" else "TELEOP"
    elif command in saved_ids:
        mode = "SAVE_POSE"
    print(f"명령 {command} → {mode}")

명령 1 → TELEOP
명령 2 → IDLE
명령 2 → TELEOP
명령 3 → SAVE_POSE
명령 0 → IDLE


## §21. 상대 모드 준비: URDF → KDL 체인

**원문 1106~1190행**

Leader와 Follower URDF를 각각 KDL Tree로 바꾼 뒤, 기준 링크에서 손끝 링크까지 Chain을 추출합니다. 두 장치의 형상이 달라도 각자의 모델로 손끝 자세를 계산할 수 있습니다.

`build_chain` 람다는 출력 인자를 참조로 받습니다. 고정 관절은 건너뛰고 움직이는 관절을 그룹 배열 인덱스와 연결합니다. `std::move`는 여기서 vector와 스마트 포인터의 불필요한 복사를 줄이는 데 사용됩니다.

```cpp
// [학습] 상대 모드 준비: URDF 트리에서 base→손끝의 직렬 관절 체인을 추출합니다.
bool JointTrajectoryCommandBroadcaster::configure_relative_pose_kinematics(
  const urdf::Model & leader_model, const urdf::Model & follower_model)
{
  group_pose_kinematics_.clear();
  // [학습] KDL Tree는 전체 구조, Chain은 두 링크 사이의 경로입니다.
  KDL::Tree leader_tree;
  if (!kdl_parser::treeFromUrdfModel(leader_model, leader_tree)) {
    RCLCPP_ERROR(get_node()->get_logger(), "Failed to build KDL tree from leader URDF");
    return false;
  }
  KDL::Tree follower_tree;
  if (!kdl_parser::treeFromUrdfModel(follower_model, follower_tree)) {
    RCLCPP_ERROR(get_node()->get_logger(), "Failed to build KDL tree from follower URDF");
    return false;
  }

  for (const auto & group_item : group_joint_names_) {
    const std::string & group_name = group_item.first;
    const std::vector<std::string> & group_joints = group_item.second;
    const std::string end_effector = group_name == "left" ?
      params_.left_end_effector_link : params_.right_end_effector_link;
    // [학습] 그룹마다 왼쪽/오른쪽 손끝 링크를 선택합니다.
    KDL::Chain leader_chain;
    KDL::Chain follower_chain;
    std::vector<size_t> leader_indices;
    std::vector<size_t> follower_indices;
    const auto build_chain =
      [this, &group_name, &group_joints, &end_effector](
      // [학습] 현재 함수 안에서만 쓰는 람다입니다. 지역 변수들을 참조 캡처해 중복 코드를 줄입니다.
      const KDL::Tree & tree, const char * role, KDL::Chain & chain,
      std::vector<size_t> & group_joint_indices) -> bool
      // [학습] chain과 group_joint_indices는 출력용 참조입니다. -> bool은 람다의 반환형 표기입니다.
      {
        if (!tree.getChain(params_.relative_base_link, end_effector, chain)) {
          RCLCPP_ERROR(
            get_node()->get_logger(), "[%s] Cannot build %s kinematic chain %s -> %s",
            group_name.c_str(), role, params_.relative_base_link.c_str(), end_effector.c_str());
          return false;
        }
        for (unsigned int segment_index = 0; segment_index < chain.getNrOfSegments();
          ++segment_index)
        {
          const KDL::Joint & joint = chain.getSegment(segment_index).getJoint();
          if (joint.getType() == KDL::Joint::None) {
            continue;
          // [학습] 고정 관절은 움직이는 자유도가 아니므로 매핑에서 제외합니다.
          }
          const auto group_joint =
            std::find(group_joints.begin(), group_joints.end(), joint.getName());
          if (group_joint == group_joints.end()) {
            // [학습] KDL 순서의 관절이 그룹 배열의 몇 번째인지 이름으로 찾습니다.
            RCLCPP_ERROR(
              get_node()->get_logger(), "[%s] %s kinematic joint '%s' is absent from group joints",
              group_name.c_str(), role, joint.getName().c_str());
            return false;
          }
          group_joint_indices.push_back(
            static_cast<size_t>(std::distance(group_joints.begin(), group_joint)));
        }
        if (group_joint_indices.size() != chain.getNrOfJoints() || chain.getNrOfJoints() == 0U) {
          RCLCPP_ERROR(
            // [학습] 반복자 사이 거리로 배열 인덱스를 구하고 size_t로 변환합니다.
            get_node()->get_logger(),
            "[%s] Invalid %s relative-pose chain (%u KDL joints, %zu mapped)",
            group_name.c_str(), role, chain.getNrOfJoints(), group_joint_indices.size());
          return false;
        }
        return true;
      };
    if (!build_chain(leader_tree, "leader", leader_chain, leader_indices) ||
      !build_chain(follower_tree, "follower", follower_chain, follower_indices))
    {
      group_pose_kinematics_.clear();
      return false;
    }

    auto kinematics =
      std::make_shared<GroupPoseKinematics>(leader_chain, follower_chain);
    kinematics->leader.group_joint_indices = std::move(leader_indices);
    // [학습] 계산기 묶음을 생성하고 인덱스 vector를 이동 전달합니다.
    kinematics->follower.group_joint_indices = std::move(follower_indices);
    group_pose_kinematics_[group_name] = std::move(kinematics);
    RCLCPP_INFO(
      get_node()->get_logger(),
      "[%s] Relative-pose kinematics configured: leader %s -> %s (%u joints), "
      "follower %s -> %s (%u joints)",
      group_name.c_str(), params_.relative_base_link.c_str(), end_effector.c_str(),
      leader_chain.getNrOfJoints(), params_.relative_base_link.c_str(), end_effector.c_str(),
      follower_chain.getNrOfJoints());
  }
  return !group_pose_kinematics_.empty();
}

```

## §22. 상대 모드 계산 ① FK·손끝 목표·Jacobian

**원문 1191~1264행**

입력은 Leader 기준 관절 배열, 제어 주기, 그룹명입니다. Leader 자세는 실제 Leader 입력으로 계산하지만 Follower의 `current_pose`는 **마지막 명령 current_action을 모델에 넣어 계산**합니다. 매 주기 follower 실측값으로 FK하는 코드가 아닙니다.

기준점을 캡처한 뒤 Leader의 상대 이동·회전을 Follower 목표로 옮깁니다. 손끝의 위치·방향 오차에 이득을 곱해 원하는 6차원 속도를 만들고, Jacobian과 감쇠 최소제곱으로 관절 속도를 구합니다.

처음에는 “손끝을 이 방향으로 움직이려면 각 관절을 얼마나 움직여야 하는가”를 계산한다고 이해하면 충분합니다.

```cpp
// [학습] 상대 모드 한 주기 계산입니다. candidate 출력 참조에 새 관절 목표를 기록합니다.
bool JointTrajectoryCommandBroadcaster::update_relative_pose_target(
  const std::string & group_name,
  const std::vector<double> & leader_reference,
  const double period_sec,
  std::vector<double> & candidate)
{
  const auto kinematics_item = group_pose_kinematics_.find(group_name);
  const auto target_item = group_last_target_.find(group_name);
  if (kinematics_item == group_pose_kinematics_.end() || !kinematics_item->second ||
    // [학습] 포인터·배열 크기·period 유효성을 먼저 확인합니다.
    target_item == group_last_target_.end() || leader_reference.size() != target_item->second.size() ||
    !std::isfinite(period_sec) || period_sec <= 0.0)
  {
    return false;
  }
  auto & kinematics = *kinematics_item->second;
  const std::vector<double> & current_action = target_item->second;
  // [학습] 여기 current_action은 마지막 명령입니다. 매 주기의 follower 실측 관절값이 아닙니다.
  KDL::JntArray current_joints;
  Eigen::Isometry3d leader_pose;
  Eigen::Isometry3d current_pose;
  if (!compute_pose(kinematics.leader, leader_reference, leader_pose) ||
    !compute_pose(kinematics.follower, current_action, current_pose, &current_joints))
  // [학습] Leader FK와 Follower 모델에 마지막 명령을 넣은 FK를 각각 계산합니다.
  {
    return false;
  }

  if (!kinematics.relative_mapper.anchored()) {
    if (!kinematics.relative_mapper.capture(leader_pose, current_pose)) {
      return false;
    // [학습] 첫 계산에서 두 손끝 자세를 기준점으로 캡처합니다.
    }
    RCLCPP_INFO(
      get_node()->get_logger(), "[%s] Relative end-effector anchors captured",
      group_name.c_str());
  }

  Eigen::Isometry3d goal;
  if (!kinematics.relative_mapper.map(leader_pose, goal)) {
    return false;
  }

  // [학습] Leader 기준점 대비 변화를 Follower 기준점에 적용해 손끝 목표를 구합니다.
  Eigen::Matrix<double, 6, 1> desired_velocity;
  desired_velocity.head<3>() =
    params_.relative_kp_position * (goal.translation() - current_pose.translation());
  desired_velocity.tail<3>() = params_.relative_kp_orientation *
    shortest_orientation_error(goal.linear(), current_pose.linear());
// [학습] 앞 3개는 위치오차 기반 선속도, 뒤 3개는 방향오차 기반 각속도입니다.

  KDL::Jacobian kdl_jacobian(kinematics.follower.chain.getNrOfJoints());
  if (kinematics.follower.jacobian_solver->JntToJac(current_joints, kdl_jacobian) < 0) {
    return false;
  }
  const Eigen::Index joint_count =
    static_cast<Eigen::Index>(kinematics.follower.chain.getNrOfJoints());
  // [학습] Jacobian은 관절 속도가 손끝 속도를 어떻게 만드는지 나타내는 6×관절수 행렬입니다.
  Eigen::MatrixXd jacobian(6, joint_count);
  for (Eigen::Index row = 0; row < 6; ++row) {
    for (Eigen::Index col = 0; col < joint_count; ++col) {
      jacobian(row, col) = kdl_jacobian(
        static_cast<unsigned int>(row), static_cast<unsigned int>(col));
    }
  }
  Eigen::Matrix<double, 6, 1> weights;
  weights.head<3>().setConstant(params_.relative_weight_position);
  weights.tail<3>().setConstant(params_.relative_weight_orientation);
  Eigen::VectorXd joint_velocity;
  if (!solve_weighted_damped_velocity(
      jacobian, desired_velocity, weights, params_.relative_damping_weight, joint_velocity))
  // [학습] 위치와 방향 오차의 중요도를 서로 다르게 설정합니다.
  {
    return false;
  }

  // [학습] 가중 감쇠 최소제곱 문제를 풀어 손끝 목표 속도에 대응하는 관절 속도를 구합니다.
  candidate = current_action;
  const auto & lower = group_lower_limits_[group_name];
  const auto & upper = group_upper_limits_[group_name];
  const auto & velocity_limit = group_velocity_limits_[group_name];
  std::vector<bool> controlled(candidate.size(), false);
  for (Eigen::Index i = 0; i < joint_count; ++i) {
```

## §23. 상대 모드 계산 ② 적분·한계·보조 관절

**원문 1265~1312행**

관절 속도에 URDF 속도 한계를 적용하고 `q_next=q_previous+period_sec*qdot`로 다음 위치 목표를 만듭니다. 위치 한계도 적용합니다.

손끝 체인에 포함되지 않은 그리퍼 등은 별도로 처리합니다. `controlled` 배열은 IK에서 이미 계산한 관절을 표시합니다. 최종 candidate의 모든 값이 유한한지 확인합니다.

```cpp
    // [학습] 기존 목표를 복사한 후 계산 대상 관절만 갱신합니다.
    const size_t group_index =
      kinematics.follower.group_joint_indices[static_cast<size_t>(i)];
    controlled[group_index] = true;
    double velocity = joint_velocity[i];
    if (group_index < velocity_limit.size() && std::isfinite(velocity_limit[group_index])) {
      // [학습] URDF 속도 한계가 유한하면 관절 속도를 제한합니다.
      velocity = std::clamp(
        velocity, -velocity_limit[group_index], velocity_limit[group_index]);
    }
    // [학습] 시간 적분: 다음 목표 위치 = 이전 목표 위치 + 주기 길이 × 관절 속도.
    double position = current_action[group_index] + period_sec * velocity;
    if (group_index < lower.size() && group_index < upper.size()) {
      position = std::clamp(position, lower[group_index], upper[group_index]);
    }
    candidate[group_index] = position;
  }

  // [학습] 손끝 체인 밖의 보조 관절을 별도 처리합니다. 그리퍼는 클러치 기준을 사용합니다.
  // Cyclo treats grippers as auxiliary references, outside RelativePoseMode.
  // Keep the final executed gripper action until the leader moves outside the
  // clutch deadband; other auxiliary joints follow the leader directly.
  const auto runtime_item = group_runtime_.find(group_name);
  if (runtime_item == group_runtime_.end()) {
    return false;
  }
  const auto & clutch = runtime_item->second.gripper_clutch;
  for (size_t i = 0; i < candidate.size(); ++i) {
    if (controlled[i]) {
      continue;
    // [학습] controlled가 참인 관절은 이미 IK로 처리했으므로 건너뜁니다.
    }
    double position = leader_reference[i];
    if (is_gripper_joint(group_joint_names_[group_name][i])) {
      if (!clutch.anchored || clutch.leader_anchor.size() != candidate.size() ||
        clutch.command_anchor.size() != candidate.size() ||
        !try_map_gripper_clutch_target(
          leader_reference[i], clutch.leader_anchor[i], clutch.command_anchor[i],
          params_.gripper_clutch_deadband, params_.gripper_clutch_scale, position))
      {
        return false;
      }
    }
    if (i < lower.size() && i < upper.size()) {
      position = std::clamp(position, lower[i], upper[i]);
    }
    candidate[i] = position;
  }
  return std::all_of(candidate.begin(), candidate.end(), [](const double value) {
           return std::isfinite(value);
    });
}

```

## §24. 기준 초기화와 follower 동기화

**원문 1313~1360행**

상대 자세 기준과 그리퍼 기준을 지우는 함수 두 개, follower 측정값으로 내부 목표를 갱신하는 함수입니다.

`sync_follower_target`은 기본적으로 팔은 실측값에 맞추고 그리퍼는 마지막 명령을 보존할 수 있습니다. 시작 시 목표가 비어 있거나 보존 옵션을 끈 경우에는 전체 실측값을 씁니다. executed-action 버퍼가 있으면 그 값을 그리퍼 보존의 근거로 우선 사용합니다.

```cpp
// [학습] 상대 기준점을 지워 다음 진입 때 새 자세를 캡처하게 합니다.
void JointTrajectoryCommandBroadcaster::reset_relative_pose_anchor(
  const std::string & group_name)
{
  const auto item = group_pose_kinematics_.find(group_name);
  if (item != group_pose_kinematics_.end() && item->second) {
    item->second->relative_mapper.reset();
  }
}

// [학습] 그리퍼 기준 배열과 anchored 플래그를 초기화합니다.
void JointTrajectoryCommandBroadcaster::reset_gripper_clutch(
  const std::string & group_name)
{
  auto & clutch = group_runtime_[group_name].gripper_clutch;
  clutch.leader_anchor.clear();
  clutch.command_anchor.clear();
  clutch.anchored = false;
}

// [학습] Follower 실측값과 마지막 그리퍼 명령을 조합해 내부 목표를 갱신합니다.
bool JointTrajectoryCommandBroadcaster::sync_follower_target(
  const std::string & group_name, const bool preserve_commanded_grippers)
{
  auto buffer_it = group_follower_state_buffers_.find(group_name);
  if (buffer_it == group_follower_state_buffers_.end() || !buffer_it->second) {
    return false;
  }
  const auto * follower_positions = buffer_it->second->readFromRT();
  if (!follower_positions || follower_positions->empty()) {
    return false;
  }
  auto & held_action = group_last_target_[group_name];
  // [학습] 초기 시작이거나 보존을 요청하지 않으면 전체 목표를 실측값으로 교체합니다.
  if (held_action.empty() || !preserve_commanded_grippers) {
    // At process startup there is no command provenance yet. Explicit callers
    // may also request a complete measured-state rebase, including grippers.
    return try_store_finite_action(*follower_positions, held_action);
  }

  const std::vector<double> * commanded_action = &held_action;
  // [학습] 기본은 기존 명령, 최종 실행 명령 버퍼가 있으면 그것을 그리퍼 보존의 근거로 사용합니다.
  auto executed_it = group_executed_action_buffers_.find(group_name);
  if (executed_it != group_executed_action_buffers_.end() && executed_it->second) {
    const auto * executed_action = executed_it->second->readFromRT();
    if (executed_action && !executed_action->empty()) {
      commanded_action = executed_action;
    }
  }
  return try_rebase_measured_action_preserving_grippers(
    // [학습] 팔 관절은 측정값, 그리퍼 관절은 명령값을 유지하는 보조 함수입니다.
    *follower_positions, *commanded_action, group_joint_names_[group_name], held_action);
}

```

## §25. TELEOP 진입 준비와 절대 모드 blend

**원문 1361~1393행**

`prepare_teleop_entry`는 기준점을 초기화합니다. 상대 모드는 그 기준점으로 시작하므로 절대 관절값 연결 blend를 끕니다. 절대 모드에서는 `start_teleop_blend`가 출발 목표와 시간을 저장합니다.

이 함수들에서 반복문으로 이동을 완료하지 않습니다. 상태만 저장하고 돌아오며, 이후 update가 여러 번 호출되면서 진행됩니다.

```cpp
// [학습] TELEOP 진입 시 상대 기준과 그리퍼 기준을 리셋합니다.
void JointTrajectoryCommandBroadcaster::prepare_teleop_entry(
  const std::string & group_name)
{
  auto & rt = group_runtime_[group_name];
  reset_relative_pose_anchor(group_name);
  reset_gripper_clutch(group_name);
  // [학습] 상대 모드는 기준점 방식이므로 절대 관절값으로 연결하는 blend를 사용하지 않습니다.
  if (teleop_reference_mode_ == TeleopReferenceMode::RELATIVE) {
    rt.blend.active = false;
    return;
  }
  start_teleop_blend(group_name);
}

// [학습] 절대 모드 진입 보간의 출발 자세와 시작 시간을 저장합니다. 여기서 직접 발행하지 않습니다.
void JointTrajectoryCommandBroadcaster::start_teleop_blend(
  const std::string & group_name)
{
  auto last_it = group_last_target_.find(group_name);
  if (last_it == group_last_target_.end() || last_it->second.empty() ||
    !std::all_of(last_it->second.begin(), last_it->second.end(), [](double value) {
      return std::isfinite(value);
    }))
  {
    return;
  }
  auto & bs = group_runtime_[group_name].blend;
  // [학습] vector 대입으로 현재 목표를 복사합니다. 이후 원본이 변해도 출발점은 유지됩니다.
  bs.start_pos = last_it->second;   // snapshot
  bs.start_time = get_node()->now();
  bs.active = true;
  RCLCPP_INFO(get_node()->get_logger(),
    "[%s] Start teleop blend (%.1fs)",
    group_name.c_str(), params_.leader_blend_duration);
}

```

## §26. 저장 자세 보간 준비와 플러그인 등록

**원문 1394~1432행**

저장 자세 이동은 출발 목표·도착 목표·시작 시각·duration을 저장하면서 시작됩니다. 목표 개수와 유한성은 확인하지만 이 함수에서 목표 관절 한계를 검증하지는 않습니다.

파일 끝의 `PLUGINLIB_EXPORT_CLASS`는 로더가 클래스를 찾도록 등록하는 매크로입니다. 실제 플러그인 이름과 라이브러리 연결은 plugin XML과 CMake 설정도 함께 담당합니다. 이 파일에 main()이 없는 이유가 여기 있습니다.

```cpp
// [학습] 저장 자세 이동의 준비 함수입니다. 실제 보간 계산은 update의 SAVE_POSE case에서 수행합니다.
bool JointTrajectoryCommandBroadcaster::start_save_pose_interp(
  const std::string & group_name, const std::vector<double> & target)
{
  auto last_it = group_last_target_.find(group_name);
  if (last_it == group_last_target_.end() || last_it->second.empty() ||
    !std::all_of(last_it->second.begin(), last_it->second.end(), [](double value) {
      return std::isfinite(value);
    }))
  {
    RCLCPP_WARN(get_node()->get_logger(),
      "[%s] last_target not initialized; skip interpolation", group_name.c_str());
    return false;
  }
  // [학습] 길이가 같고 모든 값이 유한해야 합니다. 이 함수에서 위치 한계 검사까지 하는 것은 아닙니다.
  if (target.size() != last_it->second.size() ||
    !std::all_of(target.begin(), target.end(), [](double value) {
      return std::isfinite(value);
    }))
  {
    RCLCPP_WARN(get_node()->get_logger(),
      "[%s] save pose is invalid (size=%zu, expected=%zu)",
      group_name.c_str(), target.size(), last_it->second.size());
    return false;
  }
  auto & st = group_runtime_[group_name].interp;
  st.start_time = get_node()->now();
  // [학습] 참조로 그룹별 보간 상태를 수정합니다.
  st.start_pos = last_it->second;
  st.target_pos = target;
  st.duration_sec = params_.save_pose_duration;
  st.active = true;
  return true;
}

}  // namespace joint_trajectory_command_broadcaster

#include "pluginlib/class_list_macros.hpp"
// [학습] 이 매크로는 클래스를 pluginlib 로더에 등록합니다. main()을 대신 작성한 줄은 아닙니다.

PLUGINLIB_EXPORT_CLASS(
  joint_trajectory_command_broadcaster::JointTrajectoryCommandBroadcaster,
  controller_interface::ControllerInterface)
```

## 보충: 상대 모드를 수식으로 읽기

보조 헤더 `relative_pose_mapper.hpp`에서 확인한 규칙입니다. `p`는 위치 벡터, `R`은 회전 행렬, `0`은 TELEOP 시작 기준을 뜻합니다.

```text
Follower 목표 위치 = Follower 시작 위치 + (Leader 현재 위치 − Leader 시작 위치)
Follower 목표 회전 = (Leader 현재 회전 × Leader 시작 회전의 전치) × Follower 시작 회전
```

이 구현은 기준 좌표축에서 이동과 회전을 각각 처리합니다. 하나의 동차변환을 `Follower0 × inverse(Leader0) × Leader현재`로 곱하는 규칙이라고 바꾸어 설명하면 정확하지 않습니다.

손끝 속도와 관절 속도의 관계는 `v ≈ J qdot`입니다. 보조 함수는 다음 선형식을 풀어 관절 속도를 얻습니다.

```text
(Jᵀ W J + d I) qdot = Jᵀ W v
```

`W`는 위치·방향의 가중치, `d`는 감쇠 가중치입니다. 구현은 역행렬을 직접 만들지 않고 LDLT 분해로 선형식을 풉니다. 이후 관절 속도와 위치를 제한하므로 목표 손끝 속도를 항상 정확히 달성하는 것은 아닙니다.

### 계산 실습 3 — 상대 위치와 한 주기 적분

회전은 제외하고 x축만 계산합니다. 좌표축이 대응하는 모델이라는 전제의 수치 예제입니다.

In [3]:
leader_start = 0.20
leader_now = 0.25
follower_start = 0.50
follower_goal = follower_start + (leader_now - leader_start)
print(f"Leader 이동: {leader_now - leader_start:.2f} m")
print(f"Follower 손끝 목표: {follower_goal:.2f} m")

previous_joint_target = 0.30
joint_velocity = 0.20
period_sec = 0.01
next_joint_target = previous_joint_target + period_sec * joint_velocity
print(f"관절 한 주기 목표: {next_joint_target:.3f} rad")
assert abs(follower_goal - 0.55) < 1e-12
assert abs(next_joint_target - 0.302) < 1e-12

Leader 이동: 0.05 m
Follower 손끝 목표: 0.55 m
관절 한 주기 목표: 0.302 rad


## 보충: 그리퍼 클러치와 마지막 명령 보존

물체를 잡았을 때 `측정 위치`는 접촉 때문에 `명령 위치`와 다를 수 있습니다. 측정 위치를 무조건 새 목표로 삼으면 그리퍼의 기존 위치 오차가 사라져 유지 동작이 달라질 수 있습니다. 이 코드가 executed-action을 따로 읽는 이유입니다. 이는 직접 힘 제어를 구현했다는 뜻은 아닙니다.

`leader_action_latch.hpp`에서 확인한 클러치 규칙은 다음과 같습니다.

```text
delta = Leader 현재 그리퍼 − TELEOP 시작 시 Leader 그리퍼
active_delta = sign(delta) × max(0, abs(delta) − deadband)
새 목표 = 시작 시 그리퍼 명령 + scale × active_delta
```

시작 시 Leader와 Follower 그리퍼 값이 다르더라도 작은 움직임에는 기존 명령을 유지합니다. 데드밴드 밖의 움직임만 연속적으로 더합니다.

### 계산 실습 4 — 데드밴드 안에서는 그리퍼 목표 유지

In [4]:
import math
leader_anchor = 0.4
command_anchor = 0.1
deadband = 0.02
scale = 1.0
for leader_now in (0.40, 0.41, 0.45, 0.35):
    delta = leader_now - leader_anchor
    active_delta = math.copysign(max(0.0, abs(delta)-deadband), delta)
    target = command_anchor + scale*active_delta
    print(f"Leader={leader_now:.2f} → 그리퍼 목표={target:.2f}")

Leader=0.40 → 그리퍼 목표=0.10
Leader=0.41 → 그리퍼 목표=0.10
Leader=0.45 → 그리퍼 목표=0.13
Leader=0.35 → 그리퍼 목표=0.07


## 함수 호출을 따라가 보는 세 가지 시나리오

### A. 시작 직후

1. manager가 플러그인을 생성하고 `on_init()`을 호출합니다.
2. `on_configure()`가 publisher/subscriber/버퍼를 만들고 follower 초기 상태와 URDF를 기다립니다.
3. `on_activate()`가 인터페이스 매핑과 메시지 배열을 준비합니다.
4. `update()`가 반복됩니다. 기본 IDLE인 경우 follower 상태로 내부 목표를 갱신합니다.
5. 유효한 목표가 있으면 IDLE에서도 궤적 메시지를 발행할 수 있습니다.

### B. 왼팔 TELEOP 켜기

```text
/leader/left_command에 1 수신
 → 구독 람다
 → handle_enable_msg("left", 1)
 → buffer.writeFromNonRT(command)
 → 다음 update에서 readFromRT()
 → apply_enable_msg("left", 1)
 → sync_follower_target / prepare_teleop_entry
 → 같은 update의 TELEOP 분기
 → candidate 계산 → 유효성 확인 → 발행
```

상대 모드는 기준점 캡처로 시작합니다. 절대 모드는 마지막 목표에서 Leader 목표로 연결하는 blend를 사용합니다. 그리퍼 기준은 별도로 잡습니다.

### C. 저장 자세 3 요청

```text
명령 3 수신 → 버퍼 → apply_enable_msg
 → group_save_poses_에서 ID 3 조회
 → start_save_pose_interp(): 출발·목표·시간 저장
 → 이후 update에서 s(t) 계산
 → duration 경과 → interp.active=false
 → SAVE_POSE 상태에서 마지막 목표 계속 유지
```

`start_*`라는 이름의 함수가 전체 이동을 동기적으로 끝내는 것이 아닙니다. **상태를 준비하는 함수 + 매 주기 상태를 소비하는 함수**로 나누어져 있습니다.

## 원문 주석과 실제 구현을 구분해서 읽기

아래는 실험으로 발견한 장애 목록이 아니라, 소스를 읽을 때 범위를 정확히 해석하기 위한 관찰입니다.

| 표현 또는 가정 | 코드에서 실제 확인되는 내용 |
|---|---|
| IDLE은 명령 발행 중지다 | 유효한 last_target이 있으면 공통 발행부가 실행됩니다. |
| `joint_states_stamped`는 Leader 원시 관절 측정이다 | position에 last_target을 대입합니다. 목표 성격의 값입니다. |
| update 인자의 time은 센서 취득 시각이다 | manager에서 전달된 time을 사용하며 실제 센서 timestamp와의 동일성은 이 파일로 확인되지 않습니다. |
| SAVE_POSE는 실제 도착을 확인한다 | elapsed/duration에 따른 시간 보간 종료를 사용합니다. |
| 상대 IK는 매번 follower 실측값에서 시작한다 | 마지막 명령 current_action으로 FK와 다음 목표를 계산합니다. |
| RealtimePublisher를 쓰면 update 전체가 무할당이다 | 임시 vector·메시지 생성과 복사 등이 있어 그렇게 단정할 수 없습니다. |
| `get_value()`는 없는 값이면 항상 NaN이다 | 관절 키가 없으면 map.at()에서 예외가 발생할 수 있습니다. |
| URDF 필터 옵션이 모든 출력 그룹에 적용된다 | init_joint_data의 joint_names_ 구성과 실제 출력 group_joint_names_ 사용을 구분해야 합니다. |
| 모든 한계 처리는 모든 모드에 동일하게 적용된다 | TELEOP에는 한계 처리가 있지만 SAVE_POSE 준비/보간에는 같은 clamp가 보이지 않습니다. |
| enable 버퍼는 모든 클릭 이벤트를 저장한다 | 최근 명령 전달 구조라 중간 명령 보존을 보장하지 않습니다. |
| 같은 이름의 파라미터가 있으면 기능이 적용된다 | 이 .cpp에서 실제 params_ 참조가 있는지 확인해야 합니다. 예를 들어 adaptive timing 관련 설정은 이 파일의 궤적 출력에 반영되지 않습니다. |

추가로 확인할 고급 주제: configure의 무기한 대기와 executor, 지역 bool 참조 캡처의 동시 접근, 비활성화 후 재활성화 시 데이터 복원, 입력 상태의 최신성, 한쪽 그룹만 설정했을 때 초기화 대기. 이 노트에서는 해당 동작을 변경하거나 실제 로봇으로 시험하지 않았습니다.

## 복습 문제

1. 이 파일에 `main()`이 없어도 `update()`가 실행되는 이유는?
2. `auto last = map[key]`와 `auto & last = map[key]`의 차이는?
3. `const std::vector<double> & positions`를 쓰는 이유는?
4. `[this, group_name]`에서 `group_name`은 값 캡처인가, 참조 캡처인가?
5. Follower 메시지의 name 순서를 그대로 믿지 않는 이유는?
6. `case Mode::IDLE`의 `break` 뒤에 publish가 실행될 수 있는 이유는?
7. `candidate`와 `last_target`을 나누는 이유는?
8. `command=2`에 디바운스를 적용하는 이유는?
9. `SAVE_POSE`는 어떤 조건으로 보간을 마치는가?
10. 상대 모드에서 `current_pose`를 만드는 관절값은 어디에서 오는가?
11. 그리퍼의 실측 위치와 마지막 명령을 따로 저장하는 이유는?
12. `Command` launch substitution과 이 코드의 `EnableCommand`는 같은 것인가?

### 정답과 해설

1. controller manager가 pluginlib로 클래스를 로드하고 생명주기·update 함수를 호출합니다.
2. 전자는 복사본, 후자는 원본을 가리키는 참조입니다. 참조 변경은 map에 저장된 값에 반영됩니다.
3. 배열 전체의 불필요한 복사를 피하고 함수가 그 인자를 수정하지 않도록 합니다.
4. 값 캡처입니다. `[&group_name]`이면 참조 캡처입니다. `this`는 현재 객체 포인터를 사용합니다.
5. ROS 메시지의 관절 순서와 설정된 그룹 순서가 다를 수 있습니다. 이름으로 대응해야 합니다.
6. break는 switch만 끝내고 그 아래의 공통 발행 코드로 진행하기 때문입니다.
7. 전체 후보가 유효할 때만 목표를 교체해, 일부 NaN이나 크기 오류가 이전 목표를 훼손하지 않도록 합니다.
8. 토글은 상태 자체가 아니라 반전 이벤트이므로 중복 수신하면 켠 직후 다시 꺼질 수 있습니다.
9. elapsed/duration이 1에 도달했는지로 판단합니다. follower 실측 도착 판정은 없습니다.
10. group_last_target_의 마지막 명령값을 Follower 모델에 넣어 계산합니다.
11. 접촉 중인 그리퍼는 측정 위치와 명령 위치가 다를 수 있고, 기존 명령을 보존하려는 목적이 있습니다.
12. 다릅니다. launch의 Command는 외부 명령 실행 결과를 값으로 사용하는 Python 객체이고, EnableCommand는 이 C++ 클래스에서 모드 요청 번호와 값을 담는 자료형입니다.

### 스스로 설명해 볼 과제

- `update()`의 IDLE부터 공통 발행부까지 손으로 화살표를 그려 보세요.
- 왼팔 명령 1을 받았을 때 어떤 map의 어떤 값이 바뀌는지 찾아보세요.
- `position=0`과 `position=NaN`이 서로 다른 이유를 설명해 보세요.
- `start_save_pose_interp()`가 발행 함수가 아닌 이유를 설명해 보세요.
- `msg.position[i] = last_target[i]`를 근거로 출력 토픽의 의미를 설명해 보세요.

## 원문 보존 및 검증 정보

- 첨부 파일: `pasted-text.txt`
- 원문 줄 수: **1,432줄**
- 원문 SHA-256: `5d18223fbfc75be8e57e5feeff74e38c0a8a186022e769d0b1c8f64032790e9f`
- 한국어 학습 주석: **173개**
- 원문 구간: **26개**, 중복·누락 없이 원문 순서대로 수록
- 원문 코드와 원래 라이선스 주석을 유지하고 `// [학습]` 주석만 삽입
- 숫자 계산 예제 4개는 Python으로 실행하고 결과를 저장
- C++ 컴파일, ROS 실행, 실제 로봇 동작 시험은 수행하지 않음

**관련 프로젝트 파일**

```text
ffw_joint_trajectory_command_broadcaster/
  src/joint_trajectory_command_broadcaster.cpp
  src/joint_trajectory_command_broadcaster_parameters.yaml
  include/joint_trajectory_command_broadcaster/
    joint_trajectory_command_broadcaster.hpp
    relative_pose_mapper.hpp
    leader_action_latch.hpp
    toggle_command_gate.hpp
  joint_trajectory_command_plugin.xml
  CMakeLists.txt
```

설명과 주석을 이 노트북 안에 모두 포함했으므로 다른 파일을 열지 않아도 본문 학습을 이어갈 수 있습니다. 보조 파일명은 구현 근거를 추적하기 위한 안내입니다.